# Pipeline de Désagrégation — Du Bloc au Bâtiment

**Objectif :** À partir d'une adresse et des données agrégées d'un bloc, prédire les caractéristiques individuelles d'un bâtiment.

**Architecture :**
1. **Phase 1** — Géocodage + Récupération géométrie OSM + Extraction hauteur TIF + Contexte spatial
2. **Phase 2** — Prédiction LLM (Mistral) avec règles métier
3. **Phase 3** — Validation vs dataset individuel

---

## 1. Configuration et imports

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, shape
from shapely import wkt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import requests
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

DATA_ROOT = r"C:\Users\alexandre.batisse\.vscode\Projet\Projet_Stage_Scalian\data\Raw"


## 2. Phase 1A — Géocodage d'une adresse

On utilise **Nominatim** (OpenStreetMap) pour convertir une adresse en coordonnées GPS.  
Gratuit, pas de clé API requise, juste respecter le rate limiting (1 req/s).

In [781]:
def geocode_address(address: str) -> dict:
    """
    Géocode une adresse via Nominatim (OpenStreetMap).
    Retourne lat, lon, adresse formatée, et les détails.
    """
    geolocator = Nominatim(user_agent="building_disaggregation_research")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    location = geocode(address, exactly_one=True, addressdetails=True)
    
    if location is None:
        print(f"❌ Adresse non trouvée : {address}")
        return None
    
    result = {
        'address_input': address,
        'address_found': location.address,
        'lat': location.latitude,
        'lon': location.longitude,
        'details': location.raw.get('address', {}),
    }
    
    print(f"✓ Géocodage réussi")
    print(f"  Adresse : {result['address_found']}")
    print(f"  Coordonnées : ({result['lat']:.6f}, {result['lon']:.6f})")
    
    return result

In [848]:
# ── Test avec une adresse ─────────────────────
# Remplace par une vraie adresse de ton dataset pour tester
TEST_ADDRESS = "Fleher Straße 177, Düsseldorf, Germany"

geo_result = geocode_address(TEST_ADDRESS)
geo_result

✓ Géocodage réussi
  Adresse : 177, Fleher Straße, Bilk, Stadtbezirk 3, Düsseldorf, Nordrhein-Westfalen, 40223, Deutschland
  Coordonnées : (51.196450, 6.768104)


{'address_input': 'Fleher Straße 177, Düsseldorf, Germany',
 'address_found': '177, Fleher Straße, Bilk, Stadtbezirk 3, Düsseldorf, Nordrhein-Westfalen, 40223, Deutschland',
 'lat': 51.1964503,
 'lon': 6.7681039,
 'details': {'house_number': '177',
  'road': 'Fleher Straße',
  'suburb': 'Bilk',
  'city_district': 'Stadtbezirk 3',
  'city': 'Düsseldorf',
  'state': 'Nordrhein-Westfalen',
  'ISO3166-2-lvl4': 'DE-NW',
  'postcode': '40223',
  'country': 'Deutschland',
  'country_code': 'de'}}

## INITIALISATION AUTOMATIQUE PAR VILLE
Fonction pour créer city_config et building_stats, informations nécéssaire au dynamique prompting
Associé à la ville traiter le bon folder de datas

In [813]:
# =============================================================================
# ANALYSE AUTOMATIQUE DU DATASET — Génère CITY_CONFIG et BUILDING_STATS
# =============================================================================

def analyze_dataset(df: pd.DataFrame, city_name: str = "Unknown") -> tuple[dict, dict]:
    """
    Analyse le dataset measured et génère automatiquement :
    - CITY_CONFIG : consommations spécifiques, heating distribution, ratios HS/FA
    - BUILDING_STATS : statistiques détaillées par building_type (seuils, distributions)
    
    Args:
        df: DataFrame du dataset measured (individuel)
        city_name: nom de la ville
    
    Returns:
        (city_config, building_stats)
    """
    
    print(f"{'=' * 60}")
    print(f"ANALYSE AUTOMATIQUE DU DATASET — {city_name}")
    print(f"{'=' * 60}")
    print(f"  {len(df)} bâtiments dans le dataset")
    
    # ── 1. Consommations spécifiques ───────────────────────────
    print("\n── Consommations spécifiques ──")
    df_a = df.copy()
    df_a['conso_specifique'] = df_a['initial_heat_demand'] / df_a['heated_space']
    df_a = df_a[(df_a['conso_specifique'] > 10) & (df_a['conso_specifique'] < 500)]
    
    conso_spec = {}
    for cyc in sorted(df_a['construction_year_class'].dropna().unique()):
        conso_spec[cyc] = {}
        for rs in ['not_renovated', 'partially_renovated', 'renovated']:
            subset = df_a[(df_a['construction_year_class'] == cyc) & (df_a['renovation_state'] == rs)]
            if len(subset) > 0:
                conso_spec[cyc][rs] = int(round(subset['conso_specifique'].median()))
            else:
                # Fallback : médiane globale de la classe
                fallback = df_a[df_a['construction_year_class'] == cyc]['conso_specifique'].median()
                conso_spec[cyc][rs] = int(round(fallback)) if pd.notna(fallback) else 100
        print(f"  {cyc}: NR={conso_spec[cyc]['not_renovated']}, PR={conso_spec[cyc]['partially_renovated']}, R={conso_spec[cyc]['renovated']}")
    
    # ── 2. Heating distribution par building_type ──────────────
    print("\n── Distribution heating system ──")
    heating_dist = {}
    if 'initial_heating_system' in df.columns:
        ct = pd.crosstab(df['building_type'], df['initial_heating_system'], normalize='index')
        for bt in ct.index:
            row = ct.loc[bt]
            top = row[row > 0.01].sort_values(ascending=False)
            heating_dist[bt] = {sys: int(round(pct * 100)) for sys, pct in top.head(6).items()}
            print(f"  {bt}: {heating_dist[bt]}")
    
    # ── 3. Ratio HS/FA par building_type ───────────────────────
    print("\n── Ratio heated_space / floor_area ──")
    df_a['ratio_hs_fa'] = df_a['heated_space'] / df_a['floor_area']
    hs_fa_ratio = {}
    for bt in df_a['building_type'].dropna().unique():
        subset = df_a[df_a['building_type'] == bt]
        ratio = subset['ratio_hs_fa'].median()
        hs_fa_ratio[bt] = round(ratio, 2) if pd.notna(ratio) else 1.0
        print(f"  {bt}: {hs_fa_ratio[bt]}")
    
    # ── 4. BUILDING_STATS — statistiques détaillées ────────────
    print("\n── Statistiques par building_type ──")
    building_stats = {}
    
    for bt in sorted(df['building_type'].dropna().unique()):
        subset = df[df['building_type'] == bt]
        stats = {
            'count': len(subset),
            'share_pct': round(len(subset) / len(df) * 100, 1),
        }
        
        # Stats pour chaque variable numérique clé
        for col in ['floor_area', 'heated_space', 'initial_heat_demand', 'construction_year']:
            if col in subset.columns:
                vals = subset[col].dropna()
                if len(vals) > 0:
                    stats[col] = {
                        'min': round(float(vals.min()), 1),
                        'p5': round(float(vals.quantile(0.05)), 1),
                        'p25': round(float(vals.quantile(0.25)), 1),
                        'median': round(float(vals.median()), 1),
                        'p75': round(float(vals.quantile(0.75)), 1),
                        'p95': round(float(vals.quantile(0.95)), 1),
                        'max': round(float(vals.max()), 1),
                    }
        
        # Stats pour construction_year_class (distribution)
        if 'construction_year_class' in subset.columns:
            cyc_dist = subset['construction_year_class'].value_counts(normalize=True)
            stats['construction_year_class_top3'] = {
                k: round(v * 100, 1) for k, v in cyc_dist.head(3).items()
            }
        
        # Stats pour renovation_state (distribution)
        if 'renovation_state' in subset.columns:
            rs_dist = subset['renovation_state'].value_counts(normalize=True)
            stats['renovation_state'] = {
                k: round(v * 100, 1) for k, v in rs_dist.items()
            }
        
        # Nombre d'appartements
        if 'number_of_apartments_min' in subset.columns:
            apt = subset['number_of_apartments_min'].dropna()
            if len(apt) > 0:
                stats['apartments'] = {
                    'median_min': int(apt.median()),
                    'median_max': int(subset['number_of_apartments_max'].dropna().median()) if 'number_of_apartments_max' in subset.columns else int(apt.median()),
                }
        # Ratio HS/FA comme proxy du nombre d'étages
        if 'heated_space' in subset.columns and 'floor_area' in subset.columns:
            ratio = subset['heated_space'] / subset['floor_area']
            ratio = ratio[(ratio > 0) & (ratio < 20)]
            if len(ratio) > 0:
                stats['ratio_hs_fa'] = {
                    'p25': round(float(ratio.quantile(0.25)), 2),
                    'median': round(float(ratio.median()), 2),
                    'p75': round(float(ratio.quantile(0.75)), 2),
                }
                
        building_stats[bt] = stats
        print(f"  {bt}: {stats['count']} bâtiments ({stats['share_pct']}%), "
              f"floor_area médian={stats.get('floor_area', {}).get('median', '?')}m²")
    
    # ── Construire le CITY_CONFIG ──────────────────────────────
    city_config = {
        "city_name": city_name,
        "conso_specifique": conso_spec,
        "heating_distribution": heating_dist,
        "hs_fa_ratio": hs_fa_ratio,
    }
    
    # Lister tous les systèmes de chauffage présents
    if 'initial_heating_system' in df.columns:
        all_systems = sorted(df['initial_heating_system'].dropna().unique().tolist())
        city_config['all_heating_systems'] = all_systems
        print(f"\n  Systèmes de chauffage : {all_systems}")
    
    print(f"\n{'=' * 60}")
    print(f"✓ Analyse terminée — {len(building_stats)} types de bâtiments analysés")
    
    return city_config, building_stats

In [814]:
# =============================================================================
# INITIALISATION AUTOMATIQUE PAR VILLE
# =============================================================================

def initialize_city(city_name: str = None, data_folder: str = None, 
                    geo_details: dict = None) -> tuple[pd.DataFrame, pd.DataFrame, dict, dict, str]:
    """
    Initialise automatiquement la pipeline pour une ville donnée.
    
    Détecte la ville depuis le géocodage ou le nom fourni,
    charge les datasets, lance l'analyse, et retourne tout.
    
    Args:
        city_name: nom de la ville (optionnel si geo_details fourni)
        geo_details: dict retourné par geocode_address (pour auto-détection)
    
    Returns:
        (df_individual, df_blocks, city_config, building_stats, tif_folder)
    """
    # ── Auto-détection de la ville ─────────────────────────────
    if city_name is None and geo_details:
        city_name = geo_details.get('details', {}).get('city', 
                    geo_details.get('details', {}).get('town',
                    geo_details.get('details', {}).get('municipality', 'Unknown')))
    
    if city_name is None:
        raise ValueError("Impossible de déterminer la ville. Fournis city_name ou geo_details.")
    
    print(f"{'=' * 60}")
    print(f"INITIALISATION — {city_name}")
    print(f"{'=' * 60}")
    
    # ── Trouver le dossier de la ville ─────────────────────────
    if data_folder is None:
        data_folder = r"C:\Users\alexandre.batisse\.vscode\Projet\Projet_Stage_Scalian\raw\data"
    
    city_folder = os.path.join(data_folder, city_name)
    
    if not os.path.isdir(city_folder):
        # Essayer sans accent / avec variantes
        for folder in os.listdir(data_folder):
            if city_name.lower() in folder.lower():
                city_folder = os.path.join(data_folder, folder)
                break
    
    if not os.path.isdir(city_folder):
        raise FileNotFoundError(f"Dossier non trouvé : {city_folder}")
    
    print(f"  Dossier : {city_folder}")
    
    # ── Trouver les fichiers CSV ───────────────────────────────
    csv_files = [f for f in os.listdir(city_folder) if f.endswith('.csv')]
    
    path_individual = None
    path_aggregated = None
    
    for f in csv_files:
        f_lower = f.lower()
        if 'aggregat' in f_lower or 'geo_area' in f_lower:
            path_aggregated = os.path.join(city_folder, f)
        elif 'building' in f_lower or 'twin' in f_lower:
            path_individual = os.path.join(city_folder, f)
        else:
            # Fallback : le plus gros fichier est probablement l'individuel
            if path_individual is None:
                path_individual = os.path.join(city_folder, f)
    
    print(f"  Dataset individuel : {os.path.basename(path_individual)}")
    print(f"  Dataset agrégé : {os.path.basename(path_aggregated) if path_aggregated else 'NON TROUVÉ'}")
    
    # ── Trouver le dossier TIF ─────────────────────────────────
    tif_folder = None
    for subfolder in ['height', 'tif', 'heights', 'TIF']:
        candidate = os.path.join(city_folder, subfolder)
        if os.path.isdir(candidate):
            tif_folder = candidate
            break
    
    if tif_folder:
        n_tif = len([f for f in os.listdir(tif_folder) if f.endswith('.tif')])
        print(f"  TIF : {tif_folder} ({n_tif} fichiers)")
    else:
        print(f"  ⚠️ Pas de dossier TIF trouvé")
    
    # ── Charger les datasets ───────────────────────────────────
    print(f"\n  Chargement des données...")
    df_individual = pd.read_csv(path_individual, sep = ";")
    print(f"  Dataset individuel : {len(df_individual)} bâtiments")
    
    df_blocks = None
    if path_aggregated:
        df_blocks = pd.read_csv(path_aggregated, sep = ";")
        print(f"  Dataset agrégé : {len(df_blocks)} blocs")
    
    # ── Nettoyer les données ───────────────────────────────────
    df_individual = df_individual.dropna(subset=['heated_space', 'initial_heat_demand'])
    df_individual = df_individual[(df_individual['heated_space'] > 0) & (df_individual['initial_heat_demand'] > 0)]
    print(f"  Après nettoyage : {len(df_individual)} bâtiments")
    
    # ── Analyser le dataset ────────────────────────────────────
    city_config, building_stats = analyze_dataset(df_individual, city_name)
    
    # ── Ajouter les coordonnées du centre-ville ────────────────
    CITY_CENTERS = {
        "Düsseldorf": (51.2277, 6.7735),
        "Gelsenkirchen": (51.5177, 7.0857),
    }
    center = CITY_CENTERS.get(city_name, None)
    if center:
        city_config["center_lat"] = center[0]
        city_config["center_lon"] = center[1]
    else:
        # Utiliser le centroïde des données comme approximation
        if 'geometry' in df_individual.columns:
            print(f"  ⚠️ Centre-ville non défini pour {city_name} — utilisation du centroïde des données")
        city_config["center_lat"] = 0
        city_config["center_lon"] = 0
    
    print(f"\n✓ Initialisation complète pour {city_name}")
    
    # Renommer la variable pour être générique
    CITY_CENTER = (city_config["center_lat"], city_config["center_lon"])

    return df_individual, df_blocks, city_config, building_stats, tif_folder


In [815]:
# ── Initialisation à partir du géocodage ───────────────────────

detected_city = geo_result['details'].get('city', 
                geo_result['details'].get('town',
                geo_result['details'].get('municipality', None)))

print(f"Ville détectée : {detected_city}")

df_individual, df_blocks, CITY_CONFIG, BUILDING_STATS, TIF_FOLDER = initialize_city(
    city_name=detected_city,
    data_folder=DATA_ROOT
)

# Mettre à jour les variables globales utilisées dans le profiling
CITY_CENTER = (CITY_CONFIG.get("center_lat", 0), CITY_CONFIG.get("center_lon", 0))
PATH_TIF = TIF_FOLDER

# Afficher ce qui a été créé
print(json.dumps(CITY_CONFIG, indent=2, ensure_ascii=False, default=str))
print("\n")
print(json.dumps(BUILDING_STATS, indent=2, ensure_ascii=False, default=str))

Ville détectée : Düsseldorf
INITIALISATION — Düsseldorf
  Dossier : C:\Users\alexandre.batisse\.vscode\Projet\Projet_Stage_Scalian\data\raw\Düsseldorf
  Dataset individuel : Düsseldorf_digital_twin_buildings.csv
  Dataset agrégé : Düsseldorf_digital_twin_geo_area_aggregated.csv
  TIF : C:\Users\alexandre.batisse\.vscode\Projet\Projet_Stage_Scalian\data\raw\Düsseldorf\height (240 fichiers)

  Chargement des données...
  Dataset individuel : 95876 bâtiments
  Dataset agrégé : 558 blocs
  Après nettoyage : 94057 bâtiments
ANALYSE AUTOMATIQUE DU DATASET — Düsseldorf
  94057 bâtiments dans le dataset

── Consommations spécifiques ──
  1860 - 1918: NR=105, PR=110, R=110
  1919 - 1948: NR=105, PR=117, R=120
  1949 - 1978: NR=105, PR=114, R=115
  1979 - 1986: NR=105, PR=116, R=117
  1987 - 1990: NR=105, PR=116, R=117
  1991 - 1995: NR=103, PR=106, R=107
  1996 - 2000: NR=97, PR=105, R=106
  2001 - 2004: NR=88, PR=93, R=89
  2005 - 2008: NR=91, PR=100, R=98
  2009 - 2011: NR=85, PR=84, R=93
  2

## 4. Phase 1B — Récupération de l'empreinte bâtiment via OSM

À partir des coordonnées GPS, on récupère :
- Le **polygone** (empreinte au sol) du bâtiment
- La **surface au sol** calculée depuis le polygone
- Les **métadonnées OSM** disponibles (nombre d'étages, type, etc.)

In [816]:
def get_building_footprint_osm(lat: float, lon: float, radius: int = 50) -> dict:
    """
    Récupère l'empreinte du bâtiment le plus proche des coordonnées
    via l'API Overpass (OpenStreetMap).
    """
    overpass_url = "https://overpass-api.de/api/interpreter"
    query = f"""
    [out:json][timeout:25];
    way["building"](around:{radius},{lat},{lon});
    out body;
    >;
    out skel qt;
    """
    
    # ── Requête avec retry ─────────────────────────────────────
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.get(
    overpass_url, 
    params={'data': query}, 
    timeout=30,
    headers={
        'Accept': 'application/json',
        'User-Agent': 'building_disaggregation_research/1.0'
    }
)
            # Vérifier le status code
            if response.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"  ⚠️ Rate limit Overpass — attente {wait}s (tentative {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            
            if response.status_code != 200:
                print(f"  ⚠️ Overpass HTTP {response.status_code} — tentative {attempt+1}/{max_retries}")
                print(f"     Réponse : {response.text[:200]}")
                time.sleep(5)
                continue
            
            # Vérifier que la réponse n'est pas vide
            if not response.text or len(response.text.strip()) == 0:
                print(f"  ⚠️ Réponse vide — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue
            
            data = response.json()
            break
            
        except requests.exceptions.Timeout:
            print(f"  ⚠️ Timeout Overpass — tentative {attempt+1}/{max_retries}")
            time.sleep(5)
            continue
        except Exception as e:
            print(f"  ⚠️ Erreur : {e} — tentative {attempt+1}/{max_retries}")
            time.sleep(5)
            continue
    else:
        print(f"❌ Échec après {max_retries} tentatives")
        return None
    
    if not data.get('elements'):
        print(f"❌ Aucun bâtiment trouvé dans un rayon de {radius}m")
        return None
    
    # Séparer les ways (bâtiments) et les nodes (points)
    nodes = {}
    ways = []
    
    for element in data['elements']:
        if element['type'] == 'node':
            nodes[element['id']] = (element['lon'], element['lat'])
        elif element['type'] == 'way':
            ways.append(element)
    
    if not ways:
        print("❌ Aucun bâtiment (way) trouvé")
        return None
    
    # Trouver le bâtiment le plus proche du point
    target_point = Point(lon, lat)
    best_building = None
    best_distance = float('inf')
    
    for way in ways:
        coords = []
        for node_id in way.get('nodes', []):
            if node_id in nodes:
                coords.append(nodes[node_id])
        
        if len(coords) < 3:
            continue
        
        from shapely.geometry import Polygon
        try:
            polygon = Polygon(coords)
            if not polygon.is_valid:
                polygon = polygon.buffer(0)
            
            distance = target_point.distance(polygon.centroid)
            if distance < best_distance:
                best_distance = distance
                best_building = {
                    'osm_id': way['id'],
                    'polygon': polygon,
                    'tags': way.get('tags', {}),
                    'distance_to_target': distance,
                }
        except Exception:
            continue
    
    if best_building is None:
        print("❌ Impossible de construire un polygone valide")
        return None
    
    # Calculer la surface au sol (en m²)
    gdf = gpd.GeoDataFrame(
        [{'geometry': best_building['polygon']}],
        crs='EPSG:4326'
    ).to_crs('EPSG:32632')
    
    floor_area_m2 = gdf.geometry.area.iloc[0]
    perimeter_m = gdf.geometry.length.iloc[0]
    compactness = 4 * np.pi * floor_area_m2 / (perimeter_m ** 2) if perimeter_m > 0 else 0
    
    tags = best_building['tags']
    
    result = {
        'osm_id': best_building['osm_id'],
        'floor_area_m2': round(floor_area_m2, 2),
        'perimeter_m': round(perimeter_m, 2),
        'compactness': round(compactness, 4),
        'osm_building_type': tags.get('building', 'unknown'),
        'osm_levels': tags.get('building:levels', None),
        'osm_roof_shape': tags.get('roof:shape', None),
        'osm_name': tags.get('name', None),
        'osm_all_tags': tags,
        'geometry_wkt': best_building['polygon'].wkt,
        'centroid_lat': best_building['polygon'].centroid.y,
        'centroid_lon': best_building['polygon'].centroid.x,
        'n_buildings_nearby': len(ways),
    }
    
    print(f"✓ Bâtiment trouvé (OSM ID: {result['osm_id']})")
    print(f"  Surface au sol : {result['floor_area_m2']:.1f} m²")
    print(f"  Périmètre : {result['perimeter_m']:.1f} m")
    print(f"  Compacité : {result['compactness']:.3f} (1.0 = cercle parfait)")
    print(f"  Type OSM : {result['osm_building_type']}")
    print(f"  Niveaux OSM : {result['osm_levels']}")
    print(f"  Bâtiments à proximité : {result['n_buildings_nearby']}")
    
    return result

## 5. Phase 1C — Extraction de la hauteur depuis le TIF

Si le fichier TIF de hauteur est disponible, on extrait la hauteur du bâtiment  
et on en déduit le nombre d'étages estimé.

In [849]:
def extract_height_from_tif(lat: float, lon: float, tif_folder: str) -> dict:
    """
    Extrait la hauteur du bâtiment à partir d'un dossier de tuiles TIF.
    Parcourt les tuiles pour trouver celle qui contient le point.
    
    Args:
        lat, lon: coordonnées GPS du bâtiment
        tif_folder: dossier contenant les fichiers .tif
    """
    import rasterio
    from pyproj import Transformer
    from shapely.geometry import box as shapely_box
    
    if not os.path.isdir(tif_folder):
        print(f"⚠️ Dossier TIF non trouvé : {tif_folder}")
        return None
    
    tif_files = [f for f in os.listdir(tif_folder) if f.endswith('.tif')]
    if not tif_files:
        print(f"⚠️ Aucun fichier .tif dans {tif_folder}")
        return None
    
    print(f"  {len(tif_files)} tuiles TIF disponibles")
    
    # Transformer les coordonnées GPS → UTM (EPSG:25832 comme ton code existant)
    transformer_to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:25832', always_xy=True)
    x_utm, y_utm = transformer_to_utm.transform(lon, lat)
    
    # Chercher la bonne tuile
    for tif_name in tif_files:
        tif_path = os.path.join(tif_folder, tif_name)
        
        try:
            with rasterio.open(tif_path) as src:
                bounds = src.bounds
                
                # Vérifier si le point est dans cette tuile
                if (bounds.left <= x_utm <= bounds.right and 
                    bounds.bottom <= y_utm <= bounds.top):
                    
                    print(f"  ✓ Tuile trouvée : {tif_name}")
                    
                    # Lire la hauteur au point (fenêtre 5x5 pour robustesse)
                    row, col = src.index(x_utm, y_utm)
                    
                    window_size = 5
                    half = window_size // 2
                    row_start = max(0, row - half)
                    col_start = max(0, col - half)
                    
                    window = rasterio.windows.Window(col_start, row_start, window_size, window_size)
                    data = src.read(1, window=window)
                    
                    # Filtrer nodata et valeurs invalides
                    nodata = src.nodata if src.nodata is not None else -9999
                    valid = data[(data != nodata) & (data > 0)]
                    
                    if len(valid) == 0:
                        print("  ⚠️ Pas de donnée de hauteur valide à cette position")
                        return {'height_m': None, 'estimated_floors': None, 'source': 'tif_no_data'}
                    
                    height_max = float(valid.max())
                    height_p90 = float(np.percentile(valid, 90))
                    
                    # Estimation étages (3.4m par étage, comme ton code)
                    FLOOR_HEIGHT_M = 3.4
                    estimated_floors = max(1, round(height_p90 / FLOOR_HEIGHT_M))
                    
                    result = {
                        'height_max_m': round(height_max, 2),
                        'height_p90_m': round(height_p90, 2),
                        'estimated_floors': estimated_floors,
                        'tif_file': tif_name,
                        'tif_crs': str(src.crs),
                        'tif_resolution_m': round(src.res[0], 2),
                        'source': 'tif'
                    }
                    
                    print(f"  Hauteur max : {result['height_max_m']:.1f} m")
                    print(f"  Hauteur p90 : {result['height_p90_m']:.1f} m")
                    print(f"  Étages estimés : {result['estimated_floors']}")
                    print(f"  Résolution : {result['tif_resolution_m']}m/pixel")
                    
                    return result
                    
        except Exception as e:
            continue
    
    print(f"  ⚠️ Aucune tuile ne couvre le point ({lat}, {lon})")
    return None


# ── Test ───────────────────────────────────────────────────────

if geo_result and os.path.isdir(TIF_FOLDER):
    height_data = extract_height_from_tif(geo_result['lat'], geo_result['lon'], TIF_FOLDER)
else:
    print("ℹ️ TIF non disponible — la hauteur sera estimée par OSM ou par le LLM")
    height_data = None  

  240 tuiles TIF disponibles
  ✓ Tuile trouvée : ndom50_32344_5674_1_nw_2023.tif
  Hauteur max : 13.3 m
  Hauteur p90 : 12.6 m
  Étages estimés : 4
  Résolution : 0.5m/pixel


In [381]:
df_a = df_individual.copy()
df_a['ratio_hs_fa'] = df_a['heated_space'] / df_a['floor_area']

# Pour chaque building_type, trouver la relation ratio = f(nombre d'étages estimé)
# On utilise ratio comme proxy du nombre d'étages pour l'analyse
print("FACTEUR D'UTILISATION PAR TYPE (ratio / nombre_etages_estimé)")
print("=" * 60)
for bt in sorted(df_a['building_type'].unique()):
    subset = df_a[df_a['building_type'] == bt]
    ratio = subset['ratio_hs_fa']
    ratio = ratio[(ratio > 0.5) & (ratio < 15)]
    if len(ratio) > 10:
        # Le facteur d'utilisation = ratio / (ratio.median() arrondi à l'entier = proxy étages)
        # Plus simple : regarder la distribution
        print(f"\n{bt}:")
        print(f"  ratio p10={ratio.quantile(0.1):.2f} p25={ratio.quantile(0.25):.2f} "
              f"med={ratio.median():.2f} p75={ratio.quantile(0.75):.2f} p90={ratio.quantile(0.9):.2f}")
        # Si on divise par le nombre d'étages estimé (ratio/3.4m par étage approximé)
        # Le facteur d'utilisation devrait être ~0.7-0.85

FACTEUR D'UTILISATION PAR TYPE (ratio / nombre_etages_estimé)

EFH:
  ratio p10=0.85 p25=1.49 med=1.49 p75=1.59 p90=1.73

GHD:
  ratio p10=0.85 p25=0.85 med=1.20 p75=1.49 p90=2.25

GMH:
  ratio p10=2.17 p25=2.75 med=3.33 p75=3.91 p90=4.49

HH:
  ratio p10=5.35 p25=5.65 med=5.65 p75=6.27 p90=7.63

Industrie:
  ratio p10=0.85 p25=0.85 med=0.85 p75=1.49 p90=1.49

MFH:
  ratio p10=1.49 p25=1.73 med=2.17 p75=2.75 p90=3.33

RH:
  ratio p10=1.26 p25=1.49 med=1.73 p75=1.73 p90=1.73

Öffentlich:
  ratio p10=0.85 p25=0.85 med=1.49 p75=1.80 p90=2.40


## 7. Phase 1E — Identification du bloc d'appartenance

On identifie dans quel bloc (du dataset agrégé) se trouve le bâtiment,  
en testant si le point GPS est contenu dans le polygone du bloc.

In [892]:
def identify_block(lat: float, lon: float, df_blocks: pd.DataFrame) -> dict:
    """
    Identifie le bloc (floor) contenant le bâtiment.
    Utilise la colonne geometry du dataset agrégé.
    """
    point = Point(lon, lat)
    
    # Convertir les géométries texte en objets Shapely
    for idx, row in df_blocks.iterrows():
        geom_str = row.get('geometry', '')
        if not geom_str or pd.isna(geom_str):
            continue
        
        try:
            polygon = wkt.loads(geom_str)
            if polygon.contains(point):
                # Extraire les données du bloc
                block_data = row.to_dict()
                
                print(f"✓ Bloc identifié")
                print(f"  Floor ID : {row.get('floor_area_id', '?')}")
                print(f"  Nom : {row.get('floor_area_name', '?')}")
                print(f"  Nombre de bâtiments : {row.get('building_count', '?')}")
                
                # Afficher les proportions principales
                bt_cols = [c for c in df_blocks.columns if c.startswith('building_type__')]
                if bt_cols:
                    print(f"  Répartition building_type :")
                    for col in bt_cols:
                        val = row[col]
                        if val > 0.01:
                            type_name = col.replace('building_type__', '')
                            print(f"    {type_name}: {val*100:.1f}%")
                
                return block_data
        except Exception as e:
            continue
    
    print(f"❌ Aucun bloc trouvé pour ({lat}, {lon})")
    print(f"   → Le bâtiment est peut-être en dehors des limites des blocs")
    return None

## 6. Phase 1D — Contexte spatial

On calcule des features de contexte géographique à partir des coordonnées :
- Distance au centre-ville
- Type de quartier (via OSM)
- Densité de bâtiments autour

In [866]:
from math import radians, cos, sin, asin, sqrt
from collections import Counter

def haversine(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))


def compute_spatial_context(lat: float, lon: float, block_data: dict = None) -> dict:
    """
    Calcule les features de contexte spatial enrichies pour un bâtiment.
    
    Combine 3 sources :
    - Distance au centre-ville (géométrique)
    - Densité et types des bâtiments voisins (OSM)
    - Proportions du bloc agrégé (dataset CoEnergy)
    """
    result = {}
    
    # ── 1. Distance au centre-ville ────────────────────────────
    dist_center = haversine(lat, lon, CITY_CENTER[0], CITY_CENTER[1])
    result['distance_center_km'] = round(dist_center, 2)
    
    # ── 2. Densité et profil des bâtiments voisins (OSM) ──────
    print("  Récupération des bâtiments voisins (OSM)...")
    neighbors = get_nearby_buildings_profile(lat, lon, radius=200)
    result.update(neighbors)
    
    # ── 3. Proportions du bloc (si disponible) ─────────────────
    if block_data:
        result['block_building_count'] = block_data.get('building_count', None)

        # Répartition complète building_type
        bt_cols = {k.replace('building_type__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('building_type__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_building_type_distribution'] = bt_cols
        
        # Répartition complète construction_year_class
        cyc_cols = {k.replace('construction_year_class__', ''): round(v * 100, 1) 
                    for k, v in block_data.items() 
                    if k.startswith('construction_year_class__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_construction_year_distribution'] = cyc_cols
        
        # Répartition complète heating_system
        hs_cols = {k.replace('initial_heating_system__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('initial_heating_system__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_heating_distribution'] = hs_cols
        
        # Répartition complète renovation_state
        rs_cols = {k.replace('renovation_state__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('renovation_state__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_renovation_distribution'] = rs_cols


    # ── Résumé ─────────────────────────────────────────────────
    print(f"✓ Contexte spatial calculé")
    print(f"  Distance centre-ville : {result['distance_center_km']:.2f} km")
    print(f"  Bâtiments dans 200m : {result.get('n_neighbors', '?')}")
    print(f"  Hauteur moy. voisins : {result.get('neighbor_avg_levels', '?')} niveaux")
    print(f"  Types voisins : {result.get('neighbor_building_types', '?')}")
    if block_data:
        print(f"  Répartition building_type dans le bloc : {result.get('block_building_type_distribution', '?')}")
        print(f"  Répartition construction_year_class dans le bloc : {result.get('block_construction_year_distribution', '?')}")
        
        print(f"  Bloc — type dominant : {result.get('block_dominant_building_type', '?')} "
              f"({result.get('block_dominant_building_share', 0)*100:.0f}%)")
        print(f"  Bloc — chauffage dominant : {result.get('block_dominant_heating', '?')} "
              f"({result.get('block_dominant_heating_share', 0)*100:.0f}%)")
        print(f"  Bloc — construction dominante : {result.get('block_dominant_construction_class', '?')} "
              f"({result.get('block_dominant_construction_share', 0)*100:.0f}%)")
        print(f"  Bloc — rénovation dominante : {result.get('block_dominant_renovation', '?')} "
              f"({result.get('block_dominant_renovation_share', 0)*100:.0f}%)")
    
    return result


def get_nearby_buildings_profile(lat: float, lon: float, radius: int = 200) -> dict:
    """
    Récupère le profil des bâtiments voisins via OSM :
    - Nombre de bâtiments
    - Distribution des types (residential, commercial, industrial...)
    - Nombre moyen d'étages
    """
    overpass_url = "https://overpass-api.de/api/interpreter"
    query = f"""
    [out:json][timeout:15];
    way["building"](around:{radius},{lat},{lon});
    out tags;
    """

    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = requests.post(
                overpass_url,
                data={'data': query},
                timeout=30,
                headers={'User-Agent': 'building_disaggregation_research/1.0'}
            )

            if response.status_code == 429 or response.status_code == 504:
                wait = 10 * (attempt + 1)
                print(f"  ⚠️ Overpass HTTP {response.status_code} — attente {wait}s (tentative {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            if response.status_code != 200:
                print(f"  ⚠️ Overpass HTTP {response.status_code} — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

            if not response.text or len(response.text.strip()) == 0:
                print(f"  ⚠️ Réponse vide — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

            data = response.json()
            elements = data.get('elements', [])

            if elements:
                break
            else:
                print(f"  ⚠️ Aucun élément retourné — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

        except Exception as e:
            print(f"  ⚠️ Erreur : {e} — tentative {attempt+1}/{max_retries}")
            time.sleep(5)
            continue
    else:
        print(f"  ❌ Échec après {max_retries} tentatives")
        return {'n_neighbors': None, 'neighbor_building_types': {}}

    # Compter les types de bâtiments
    building_types = []
    levels_list = []

    for el in elements:
        tags = el.get('tags', {})
        bt = tags.get('building', 'yes')
        building_types.append(bt)

        levels = tags.get('building:levels')
        if levels:
            try:
                levels_list.append(float(levels))
            except ValueError:
                pass

    # Distribution des types
    type_counts = Counter(building_types)
    total = len(building_types)
    type_distribution = {k: round(v/total, 3) for k, v in type_counts.most_common(5)}

    result = {
        'n_neighbors': len(elements),
        'neighbor_building_types': type_distribution,
        'neighbor_dominant_type': type_counts.most_common(1)[0][0] if type_counts else 'unknown',
    }

    # Hauteur moyenne si disponible
    if levels_list:
        result['neighbor_avg_levels'] = round(np.mean(levels_list), 1)
        result['neighbor_max_levels'] = int(max(levels_list))
        result['neighbor_levels_available'] = len(levels_list)
    else:
        result['neighbor_avg_levels'] = None
        result['neighbor_max_levels'] = None
        result['neighbor_levels_available'] = 0

    return result

## 8. Assemblage — Collecte complète pour un bâtiment

Fonction qui enchaîne toutes les étapes de la Phase 1  
et retourne un profil complet prêt pour le LLM.

In [891]:
def collect_building_profile(address: str, df_blocks: pd.DataFrame, tif_path: str = None) -> dict:
    """
    Pipeline complète Phase 1 : adresse → profil complet du bâtiment.
    
    Retourne un dictionnaire avec toutes les données collectées,
    prêt à être passé au LLM pour prédiction.
    """
    print("=" * 60)
    print(f"PROFIL BÂTIMENT : {address}")
    print("=" * 60)
    
    profile = {'address': address}
    
    # 1. Géocodage
    print("\n── Étape 1 : Géocodage ──")
    geo = geocode_address(address)
    if geo is None:
        return None
    profile['geocoding'] = geo
    
    # 2. Empreinte bâtiment depuis le dataset individuel
    print("\n── Étape 2 : Empreinte bâtiment (dataset individuel) ──")
    footprint = None

    street_name = geo['details'].get('road') or geo['details'].get('street') or ''
    house_number = str(geo['details'].get('house_number', '')).strip()

    if 'street' in df_individual.columns:
        candidates = df_individual.copy()

        if street_name:
            street_lower = street_name.strip().lower()
            mask = pd.Series(False, index=candidates.index)
            if house_number:
                exact_address = f"{street_lower} {house_number}".strip()
                reverse_address = f"{house_number} {street_lower}".strip()
                mask |= candidates['street'].fillna('').str.lower().str.contains(exact_address, na=False)
                mask |= candidates['street'].fillna('').str.lower().str.contains(reverse_address, na=False)
            mask |= candidates['street'].fillna('').str.lower().str.contains(street_lower, na=False)
            candidates = candidates[mask]

        if len(candidates) == 0 and 'postal_code' in df_individual.columns and geo['details'].get('postcode'):
            candidates = df_individual[
                df_individual['postal_code'].astype(str).fillna('').str.contains(str(geo['details']['postcode']), na=False)
            ]

        if len(candidates) > 1 and 'geometry' in df_individual.columns:
            point = Point(geo['lon'], geo['lat'])
            candidates = candidates.copy()
            candidates['distance_to_point'] = candidates['geometry'].apply(
                lambda geom: Point(geo['lon'], geo['lat']).distance(wkt.loads(geom).centroid)
                if pd.notna(geom) else np.inf
            )
            candidates = candidates.sort_values('distance_to_point')

        if len(candidates) > 0:
            matched = candidates.iloc[0]
            footprint = {
                'source': 'dataset_match',
                'matched_street': matched.get('street'),
                'floor_area_m2': float(matched.get('floor_area')) if pd.notna(matched.get('floor_area')) else None,
                'geometry_wkt': matched.get('geometry'),
            }

    if footprint is None:
        print("  ⚠️ Aucun bâtiment trouvé dans le dataset — fallback OSM")
        footprint = get_building_footprint_osm(geo['lat'], geo['lon'])

    profile['footprint'] = footprint
    print(f"  Surface au sol : {footprint['floor_area_m2']:.1f} m²" if footprint and footprint.get('floor_area_m2') else "  Surface au sol : N/A")
    
# 3. Hauteur TIF (si disponible)
    print("\n── Étape 3 : Hauteur (TIF) ──")
    if tif_path and os.path.isdir(tif_path):
        height = extract_height_from_tif(geo['lat'], geo['lon'], tif_path)
        profile['height'] = height
    elif tif_path and os.path.isfile(tif_path):
        # Cas d'un fichier TIF unique
        height = extract_height_from_tif(geo['lat'], geo['lon'], os.path.dirname(tif_path))
        profile['height'] = height
    else:
        # Utiliser les niveaux OSM comme fallback
        osm_levels = footprint.get('osm_levels') if footprint else None
        if osm_levels:
            estimated_height = int(osm_levels) * 3.0
            profile['height'] = {
                'height_max_m': estimated_height,
                'estimated_floors': int(osm_levels),
                'source': 'osm_levels'
            }
            print(f"  ℹ️ Hauteur estimée via OSM levels : {estimated_height}m ({osm_levels} étages)")
        else:
            profile['height'] = None
            print(f"  ⚠️ Hauteur non disponible (tif_path={tif_path})")
    
    # 4. Identification du bloc
    print("\n── Étape 4 : Identification du bloc ──")
    block = identify_block(geo['lat'], geo['lon'], df_blocks)
    profile['block_data'] = block

    # 5. Contexte spatial
    print("\n── Étape 5 : Contexte spatial ──")
    spatial = compute_spatial_context(geo['lat'], geo['lon'], block_data=block) 
    profile['spatial_context'] = spatial
    
    # 6. Résumé
    print("\n" + "=" * 60)
    print("RÉSUMÉ DU PROFIL")
    print("=" * 60)
    print(f"  Adresse : {geo['address_found']}")
    print(f"  Coordonnées : ({geo['lat']:.6f}, {geo['lon']:.6f})")
    if footprint:
        print(f"  Surface au sol (OSM) : {footprint['floor_area_m2']:.1f} m²")
    if profile.get('height'):
        print(f"  Hauteur : {profile['height']['height_max_m']:.1f} m ({profile['height']['estimated_floors']} étages)")
    print(f"  Distance centre-ville : {spatial['distance_center_km']:.2f} km")
    if block:
        print(f"  Bloc : {block.get('floor_area_name', '?')} ({block.get('building_count', '?')} bâtiments)")
    
    return profile

# ── Test complet ───────────────────────────────────────────
profile = collect_building_profile(
    address=TEST_ADDRESS,
    df_blocks=df_blocks,
    tif_path=PATH_TIF if os.path.exists(PATH_TIF) else None
)

PROFIL BÂTIMENT : Fleher Straße 177, Düsseldorf, Germany

── Étape 1 : Géocodage ──
✓ Géocodage réussi
  Adresse : 177, Fleher Straße, Bilk, Stadtbezirk 3, Düsseldorf, Nordrhein-Westfalen, 40223, Deutschland
  Coordonnées : (51.196450, 6.768104)

── Étape 2 : Empreinte bâtiment (dataset individuel) ──
  Surface au sol : 159.6 m²

── Étape 3 : Hauteur (TIF) ──
  240 tuiles TIF disponibles
  ✓ Tuile trouvée : ndom50_32344_5674_1_nw_2023.tif
  Hauteur max : 13.3 m
  Hauteur p90 : 12.6 m
  Étages estimés : 4
  Résolution : 0.5m/pixel

── Étape 4 : Identification du bloc ──
✓ Bloc identifié
  Floor ID : 3d66786a-36cf-45e6-8330-e89cb98a3495
  Nom : 17
  Nombre de bâtiments : 279
  Répartition building_type :
    EFH: 36.2%
    Industrie: 1.1%
    MFH: 22.2%
    RH: 40.1%

── Étape 5 : Contexte spatial ──
  Récupération des bâtiments voisins (OSM)...
✓ Contexte spatial calculé
  Distance centre-ville : 3.50 km
  Bâtiments dans 200m : 381
  Hauteur moy. voisins : 3.3 niveaux
  Types voisins : 

## 9. Recherche de la vérité terrain (validation)

Pour valider les prédictions, on cherche le bâtiment correspondant  
dans le dataset individuel via le `floor_area_id` et la proximité géographique.

In [798]:
def find_ground_truth(profile: dict, df_individual: pd.DataFrame) -> pd.Series:
    """
    Cherche le bâtiment correspondant dans le dataset individuel.
    
    Stratégie :
    1. Matcher par adresse (rue + numéro) — le plus fiable
    2. Affiner par floor_area si plusieurs résultats
    3. Fallback par floor_area_id + surface au sol
    """
    if profile is None:
        print("⚠️ Profil manquant")
        return None
    
    # ── Extraire rue et numéro depuis le géocodage ─────────────
    geo = profile['geocoding']
    details = geo.get('details', {})
    
    # Nominatim retourne road + house_number dans les details
    street_name = details.get('road', '')
    house_number = details.get('house_number', '')
    
    # Construire l'adresse telle qu'elle apparaît dans le dataset
    # Format dataset : "FriedrichstraÃŸe 61a"
    search_address = f"{street_name} {house_number}".strip()
    
    print(f"  Adresse recherchée : '{search_address}'")
    
    if 'street' not in df_individual.columns:
        print("⚠️ Colonne 'street' absente du dataset")
        return _fallback_by_area(profile, df_individual)
    
    # ── Normaliser pour gérer l'encodage ß / ÃŸ ───────────────
    def normalize_street(s):
        """Normalise une adresse pour la comparaison."""
        if pd.isna(s):
            return ''
        s = str(s).strip().lower()
        # Gérer les variantes d'encodage du ß
        s = s.replace('ß', 'ss')
        s = s.replace('ãŸ', 'ss')      # ÃŸ en minuscule
        s = s.replace('\u00c3\u009f', 'ss')  # bytes UTF-8 mal décodés
        # Gérer les autres caractères allemands courants
        s = s.replace('ä', 'ae').replace('ã¤', 'ae')
        s = s.replace('ö', 'oe').replace('ã¶', 'oe')
        s = s.replace('ü', 'ue').replace('ã¼', 'ue')
        # Supprimer les espaces multiples
        s = ' '.join(s.split())
        return s
    
    search_normalized = normalize_street(search_address)
    print(f"  Normalisée : '{search_normalized}'")
    
    # Normaliser la colonne street du dataset
    df_individual['_street_normalized'] = df_individual['street'].apply(normalize_street)
    
    # ── Match exact (rue + numéro) ─────────────────────────────
    exact_matches = df_individual[df_individual['_street_normalized'] == search_normalized]
    
    if len(exact_matches) == 1:
        match = exact_matches.iloc[0]
        print(f"\n✓ Match exact trouvé !")
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    if len(exact_matches) > 1:
        print(f"  {len(exact_matches)} matchs exacts — affinage par floor_area")
        match = _refine_by_floor_area(exact_matches, profile)
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    # ── Match partiel (même rue, numéro différent ou absent) ───
    street_only = normalize_street(street_name)
    number_only = house_number.strip().lower()
    
    print(f"  Pas de match exact — recherche partielle sur '{street_only}'")
    
    # Chercher tous les bâtiments de la même rue
    same_street = df_individual[
        df_individual['_street_normalized'].str.startswith(street_only)
    ]
    
    if len(same_street) > 0:
        print(f"  {len(same_street)} bâtiments trouvés dans la même rue")
        
        # Chercher le bon numéro dans la rue
        if number_only:
            with_number = same_street[
                same_street['_street_normalized'].str.contains(number_only, na=False)
            ]
            if len(with_number) > 0:
                match = _refine_by_floor_area(with_number, profile)
                print(f"\n✓ Match par rue + numéro partiel")
                _print_match(match, profile)
                df_individual.drop(columns=['_street_normalized'], inplace=True)
                return match
        
        # Fallback : même rue, plus proche par floor_area
        match = _refine_by_floor_area(same_street, profile)
        print(f"\n⚠️ Match approximatif (même rue, meilleur floor_area)")
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    # ── Dernier fallback : bloc + floor_area ───────────────────
    print("  ⚠️ Rue non trouvée — fallback par bloc + floor_area")
    df_individual.drop(columns=['_street_normalized'], inplace=True)
    return _fallback_by_area(profile, df_individual)


def _refine_by_floor_area(candidates: pd.DataFrame, profile: dict) -> pd.Series:
    """Parmi plusieurs candidats, choisir le plus proche par floor_area."""
    osm_floor_area = profile.get('footprint', {}).get('floor_area_m2')
    
    if osm_floor_area and 'floor_area' in candidates.columns:
        candidates = candidates.copy()
        candidates['_fa_diff'] = abs(candidates['floor_area'] - osm_floor_area)
        best = candidates.nsmallest(1, '_fa_diff').iloc[0]
        return best
    
    return candidates.iloc[0]


def _fallback_by_area(profile: dict, df_individual: pd.DataFrame) -> pd.Series:
    """Fallback : chercher par floor_area_id + surface au sol."""
    block_id = profile.get('block_data', {}).get('floor_area_id')
    
    if block_id and 'floor_area_id' in df_individual.columns:
        candidates = df_individual[df_individual['floor_area_id'] == block_id]
        if len(candidates) > 0:
            return _refine_by_floor_area(candidates, profile)
    
    print("❌ Aucun match trouvé")
    return None


def _print_match(match: pd.Series, profile: dict):
    """Affiche les détails du match trouvé."""
    osm_floor_area = profile.get('footprint', {}).get('floor_area_m2')
    
    print(f"  Adresse dataset : {match.get('street', '?')}")
    if osm_floor_area and 'floor_area' in match.index:
        diff = abs(match['floor_area'] - osm_floor_area)
        print(f"  floor_area → OSM: {osm_floor_area:.1f} m² | Dataset: {match['floor_area']:.1f} m² | Écart: {diff:.1f} m²")
    
    for col in ['building_type', 'heated_space', 'initial_heat_demand', 
                 'construction_year', 'initial_heating_system', 'renovation_state']:
        if col in match.index:
            print(f"  {col} : {match[col]}")

In [859]:
# ── Test : trouver la ground truth ─────────────────────────
if profile:
    ground_truth = find_ground_truth(profile, df_individual)

  Adresse recherchée : 'Fleher Straße 177'
  Normalisée : 'fleher strasse 177'

✓ Match exact trouvé !
  Adresse dataset : Fleher Straße 177
  floor_area → OSM: 159.6 m² | Dataset: 159.6 m² | Écart: 0.0 m²
  building_type : MFH
  heated_space : 201.1243871744589
  initial_heat_demand : 38066.4600296
  construction_year : 1978.0
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : partially_renovated


## 10. Prochaines étapes

La Phase 1 collecte toutes les données mesurables. La suite :

- **Phase 2** : Construire le prompt LLM avec le profil collecté + données du bloc + règles métier
- **Phase 3** : Appeler Mistral pour prédire building_type, heated_space, heat_demand, etc.
- **Phase 4** : Comparer avec la ground truth et calculer les métriques
- **Phase 5** : Itérer sur les règles métier et le prompt pour améliorer la précision

In [860]:
# ── Sauvegarde du profil pour la Phase 2 ───────────────────
if profile:
    # Nettoyer pour JSON (retirer les objets non sérialisables)
    profile_clean = json.loads(json.dumps(profile, default=str))
    
    with open('building_profile_test.json', 'w') as f:
        json.dump(profile_clean, f, indent=2, ensure_ascii=False)
    
    print("✓ Profil sauvegardé dans building_profile_test.json")

✓ Profil sauvegardé dans building_profile_test.json


## PHASE 2 Prédiction LLM En Cascade

In [ ]:
# =============================================================================
# PHASE 2 — PRÉDICTION LLM 
# =============================================================================
load_dotenv()
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── CONFIGURATION DU MODÈLE ─────────────────────────────────
  # True = Ollama local, False = API Mistral
# ── CONFIGURATION DU MODÈLE ─────────────────────────────────
LLM_PROVIDER = "claude"  # "mistral", "groq", "gemini", "local"

if LLM_PROVIDER == "mistral":
    from langchain_mistralai import ChatMistralAI
    llm = ChatMistralAI(model="mistral-medium-latest", 
                        temperature=0.0, 
                        api_key=os.getenv("MISTRAL_API_KEY"))

if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
    model="llama-3.1-8b-instant",  # ou "mixtral-8x7b-32768"
    temperature=0.0,
    api_key=os.getenv("GROQ_API_KEY"))

if LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.0,
        api_key=os.getenv("GEMINI_API_KEY"))

if LLM_PROVIDER == "local phi":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="phi4-mini", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "local mistral" :
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="mistral", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "local gemma" :
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="gemma2:9b", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "claude" :
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-2", temperature=0.0, api_key=os.getenv("CLAUDE_API_KEY"))

print(f"✓ Modèle : {LLM_PROVIDER}")


✓ Modèle : mistral


## Extraction des infos profile dans le json

In [862]:
import re

def extract_json_from_response(raw: str) -> dict:
    """
    Extrait un objet JSON d'une réponse LLM, même si le modèle
    ajoute du texte explicatif autour.
    """
    raw = raw.strip()
    
    # Cas 1 : réponse propre qui commence par {
    if raw.startswith('{'):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
    
    # Cas 2 : JSON dans des backticks markdown
    md_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if md_match:
        try:
            return json.loads(md_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Cas 3 : trouver le premier { ... } dans le texte
    brace_match = re.search(r'\{[^{}]*\}', raw)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass
    
    # Cas 4 : JSON imbriqué (pour les cas avec des nested braces)
    deep_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if deep_match:
        candidate = deep_match.group(0)
        # Nettoyer les retours à la ligne dans les valeurs
        candidate = re.sub(r'\n\s*', ' ', candidate)
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    
    # Cas 5 : extraire les valeurs manuellement par regex
    result = {}
    
    # Chercher des patterns comme "building_type": "MFH" ou building_type : MFH
    for key in ['building_type', 'type_of_use', 'construction_year_class',
                'initial_heating_system', 'renovation_state',
                'number_of_apartments_min', 'number_of_apartments_max',
                'heated_space', 'initial_heat_demand']:
        # Pattern avec guillemets
        pattern = rf'["\']?{key}["\']?\s*[:=]\s*["\']?([^"\',\n\}}]+)["\']?'
        match = re.search(pattern, raw, re.IGNORECASE)
        if match:
            val = match.group(1).strip().rstrip(',').strip('"\'')
            # Essayer de convertir en nombre si possible
            try:
                val = int(val)
            except ValueError:
                try:
                    val = float(val)
                except ValueError:
                    pass
            result[key] = val
    
    if result:
        return result
    
    print(f"    ⚠️ Impossible d'extraire du JSON : {raw[:150]}")
    return {}

## Récupération des few-shots du dataset Digital Twin pour donner des exemples de datas complètes au modèle

In [863]:
# Colonnes du dataset utilisées pour le few-shot
FEATURE_COLS_ALL = [
    'building_type', 'type_of_use', 'construction_year_class',
    'floor_area', 'heated_space', 'initial_heat_demand',
    'initial_heating_system', 'renovation_state',
    'number_of_apartments_min', 'number_of_apartments_max',
    'construction_year', 'solar_potential'
]

def select_training_examples(df: pd.DataFrame, n: int = 15, seed: int = 42) -> pd.DataFrame:
    np.random.seed(seed)
    
    examples = []
    
    # D'abord, garantir au moins 2 exemples par building_type
    for bt, group in df.groupby('building_type'):
        n_take = min(2, len(group))
        examples.append(group.sample(n=n_take, random_state=seed))
    
    # Ensuite, garantir au moins 1 exemple par construction_year_class
    base = pd.concat(examples)
    for cyc, group in df.groupby('construction_year_class'):
        if cyc not in base['construction_year_class'].values:
            examples.append(group.sample(n=1, random_state=seed))
    
    base = pd.concat(examples).drop_duplicates()
    
    # Compléter jusqu'à n
    remaining = df.loc[df.index.difference(base.index)]
    n_extra = n - len(base)
    if n_extra > 0 and len(remaining) > 0:
        extra = remaining.sample(n=min(n_extra, len(remaining)), random_state=seed)
        base = pd.concat([base, extra])
    
    result = base.head(n)
    
    print(f"✓ {len(result)} exemples sélectionnés")
    print(f"  Couverture building_type : {sorted(result['building_type'].unique())}")
    print(f"  Couverture construction_year_class : {sorted(result['construction_year_class'].unique())}")
    print(f"  Répartition building_type :")
    for bt, count in result['building_type'].value_counts().items():
        print(f"    {bt}: {count}")
    
    return result

training_examples = select_training_examples(df_individual, n=15)

✓ 15 exemples sélectionnés
  Couverture building_type : ['EFH', 'GHD', 'GMH', 'HH', 'Industrie', 'MFH', 'RH', 'Öffentlich']
  Couverture construction_year_class : ['1919 - 1948', '1949 - 1978', '1991 - 1995', '1996 - 2000', '2012 - 2023']
  Répartition building_type :
    EFH: 2
    GHD: 2
    GMH: 2
    HH: 2
    Industrie: 2
    MFH: 2
    RH: 2
    Öffentlich: 1


## SÉRIALISATION CSV (meilleur format d'après le benchmark)

In [864]:
# =============================================================================
# SÉRIALISATION CSV (meilleur format d'après le benchmark)
# =============================================================================

def serialize_example_csv(row: pd.Series, cols: list, include_values: bool = True) -> str:
    """Sérialise une ligne en format CSV."""
    values = []
    for col in cols:
        val = row.get(col, '')
        if pd.isna(val):
            values.append('')
        elif isinstance(val, float):
            values.append(f"{val:.2f}")
        else:
            values.append(str(val))
    return ",".join(values)


def build_csv_header(cols: list) -> str:
    """Construit le header CSV."""
    return ",".join(cols)

## Génération dynamique des prompts

In [865]:
# =============================================================================
# GÉNÉRATION DYNAMIQUE DES PROMPTS
# =============================================================================

def build_conso_table(config: dict) -> str:
    lines = ["| Classe | not_renovated | partially_renovated | renovated |"]
    lines.append("|--------|--------------|-------------------|-----------|")
    for classe, vals in config["conso_specifique"].items():
        lines.append(f"| {classe} | {vals.get('not_renovated', '?')} | {vals.get('partially_renovated', '?')} | {vals.get('renovated', '?')} |")
    return "\n".join(lines)


def build_heating_rules(config: dict) -> str:
    lines = []
    for bt, dist in config.get("heating_distribution", {}).items():
        if dist:
            parts = [f"{sys}: {pct}%" for sys, pct in sorted(dist.items(), key=lambda x: -x[1])]
            lines.append(f"- {bt} → {', '.join(parts)}")
    return "\n".join(lines)


def build_hs_fa_rules(config: dict) -> str:
    lines = []
    for bt, ratio in config.get("hs_fa_ratio", {}).items():
        if ratio > 0:
            lines.append(f"- {bt}: heated_space ≈ floor_area × {ratio}")
    return "\n".join(lines)


def build_building_type_rules(stats: dict) -> str:
    """Génère les règles de classification data-driven à partir de BUILDING_STATS."""
    lines = []
    for bt, s in sorted(stats.items()):
        fa = s.get('floor_area', {})
        hs = s.get('heated_space', {})
        count = s.get('count', 0)
        share = s.get('share_pct', 0)
        apt = s.get('apartments', {})      
        ratio = s.get('ratio_hs_fa', {})
        
        if not fa:
            continue
        
        line = f"{bt} ({share}% of the dataset, {count} buildings) :\n"
        line += f"  - commun floor_area : {fa.get('p25', '?')} — {fa.get('p75', '?')} m² (médian {fa.get('median', '?')}m²)\n"
        line += f"  - extreme floor_area : {fa.get('min', '?')} — {fa.get('max', '?')} m²"
        
        if hs:
            line += f"\n  - commun heated_space : {hs.get('p25', '?')} — {hs.get('p75', '?')} m² (médian {hs.get('median', '?')}m²)"
        
        if apt:
            line += f"\n  - apartments : {apt.get('median_min', '?')} — {apt.get('median_max', '?')}"

        if ratio:
            line += f"\n  - ratio commun floors (HS/FA) : {ratio.get('p25', '?')} — {ratio.get('p75', '?')} (médian {ratio.get('median', '?')})"

        lines.append(line)
    
    return "\n\n".join(lines)


def generate_prompts(city_config: dict, building_stats: dict) -> tuple[str, str, str]:
    """
    Génère les 3 system prompts dynamiquement à partir des configs.
    Retourne (prompt_level1, prompt_level2, prompt_level3).
    """
    city = city_config.get("city_name", "Unknown")
    bt_rules = build_building_type_rules(building_stats)
    heating_rules = build_heating_rules(city_config)
    conso_table = build_conso_table(city_config)
    hs_fa_rules = build_hs_fa_rules(city_config)
    
    # Liste des systèmes de chauffage
    all_systems = city_config.get('all_heating_systems', [])
    residential_systems = [s for s in all_systems if not s.startswith('Industrie')]
    industrial_systems = [s for s in all_systems if s.startswith('Industrie')]
    
    prompt_l1 = f"""You are an expert on German buildings in {city}.

    Predict 3 variables:
    1. building_type from {sorted(building_stats.keys())}
    2. type_of_use from [Wohngebäude, Gewerbegebäude, Industriegebäude, Öffentliche Hand]
    3. construction_year_class from {sorted(city_config['conso_specifique'].keys())}

    BUILDING TYPE DESCRIPTIONS:

    EFH (Einfamilienhaus — detached house):
    DETACHED residential house, few floors, only 1 housing unit
    In peripheral residential area, suburban neighborhood
    OSM signals: "house", "detached"

    RH (Reihenhaus — terraced house):
    SEMI-DETACHED residential house, attached to other houses in a row
    In residential housing developments, often aligned along a street
    Floor_area generally SMALLER than EFH (narrower house because it is semi-detached)
    OSM signals: "terrace", "semidetached_house"
    DIFFERENCE WITH EFH: the RH is semi-detached, the EFH is detached. If OSM neighbors show many detached "house" → EFH. If "semidetached" or "terrace" → RH.

    MFH (Mehrfamilienhaus — multi-family building):
    MEDIUM-sized residential building with several housing units
    The MOST COMMON type in German cities
    OSM signals: "apartments" with a medium size

    GMH (Großes Mehrfamilienhaus — large multi-family building):
    Large residential building with many housing units
    DIFFERENCE WITH MFH: floor_area and heated_space SIGNIFICANTLY larger — see the statistics below
    DIFFERENCE WITH HH: LOWER height, fewer floors

    HH (Hochhaus — high-rise / residential tower):
    Very HIGH residential tower with many floors
    VERY RARE — the building must be significantly higher than its neighbors
    DIFFERENCE WITH GMH: height is the main criterion, a HH is a real tower

    GHD (Gewerbe/Handel/Dienstleistung — commerce/services/offices):
    Building for COMMERCIAL use, no housing units
    Use: Gewerbegebäude
    Can be of any size — the criterion is the USE, not the size
    DIFFERENCE WITH residential types: if the block has a significant proportion of GHD and the building does not look like typical residential → likely GHD

    Industrie (industrial building):
    PRODUCTION or MANUFACTURING building, large volumes, few floors
    Use: Industriegebäude
    Often in industrial area with few residential buildings around
    OSM signals: "industrial", "warehouse"

    Öffentlich (public building):
    Institutional building (school, town hall, hospital, administration)
    Use: Öffentliche Hand
    OSM signals: "school", "public", "civic", "government"

    REAL STATISTICS OF {city.upper()} BY TYPE:
    {bt_rules}

    HOW TO CLASSIFY — reason in this order:
    1. OSM TAG: if the tag is specific ("house", "industrial", "school"), it is a STRONG signal
    2. NEIGHBORHOOD: the dominant type of neighboring buildings indicates the character of the district
    3. STATISTICS: compare floor_area and HS/FA ratio with typical ranges for each type
    4. BLOCK: the distribution of the block indicates which types EXIST in the area — the building can be any of them
    5. IN CASE OF DOUBT: if floor_area and height match several types, favor the one that is most frequent in the block BUT verify that other characteristics are consistent

    CONSTRUCTION_YEAR_CLASS:
    - Use the dominant class of the block as a starting point
    - A building atypical for its neighborhood (different size or height from neighbors) may be from another era

    RESPOND ONLY with this JSON, nothing else:
    {{"building_type": "...", "type_of_use": "...", "construction_year_class": "..."}}"""

    # ── LEVEL 2 ────────────────────────────────────────────────
    prompt_l2 = f"""You are an expert on building energy systems in {city}.

Predict 4 variables:
1. initial_heating_system from {all_systems}
2. renovation_state from [not_renovated, partially_renovated, renovated]
3. number_of_apartments_min (integer)
4. number_of_apartments_max (integer)

HEATING SYSTEMS — UNDERSTANDING THE DIFFERENCES:

Residential systems:
- Gaskessel (Bestand): central gas boiler, the MOST COMMON in {city}
- GEH (Bestand): INDIVIDUAL gas boiler per floor/apartment, frequent in old MFH
- Ölkessel (Bestand): oil boiler, more common in OLD buildings and in the PERIPHERY
- Fernwärme (Bestand): CENTRALIZED urban district heating network, especially in LARGE buildings in DENSE areas
- Nahwärme: LOCAL neighborhood heating network, smaller than Fernwärme
- Nachtspeicher (Bestand): electric storage heating, typical of the 1960s-1980s
- el. WP (Bestand): electric heat pump, mainly in RECENT constructions
- Pelletkessel (Bestand): pellet boiler, RARE, atypical cases

Industrial systems (ONLY for building_type = Industrie):
- Industrie Gasanlage, Industrie Kohleanlage, Industrie Stromanlage, Industrie Ölanlage, Industrie Pelletanlage
- NEVER assign an "Industrie" system to a NON-industrial building

LINKS BETWEEN VARIABLES FOR HEATING:
- OLD building (pre-1978) + PERIPHERY → higher probability of Ölkessel
- OLD building (pre-1978) + CITY CENTER → Gaskessel or Fernwärme likely
- LARGE building (GMH/HH) + DENSE area → Fernwärme more likely
- RECENT building (post-2000) → el. WP or modern Gaskessel
- The dominant heating of the BLOCK is a good indicator BUT exceptions exist

REAL DISTRIBUTION IN {city.upper()}:
{heating_rules}

RENOVATION:
- OLD building (pre-1978): often not_renovated or partially_renovated
- RECENT building (post-1996): ALMOST ALWAYS not_renovated (no need to renovate)
- The dominant renovation_state of the BLOCK is indicative

APARTMENTS:
- EFH: 1 housing unit
- RH: 1-2 housing units
- MFH: several housing units, proportional to the building size — see dataset statistics
- GMH: many housing units
- HH: very many housing units
- GHD / Industrie / Öffentlich: 0

RESPOND ONLY with this JSON, nothing else:
{{"initial_heating_system": "...", "renovation_state": "...", "number_of_apartments_min": X, "number_of_apartments_max": Y}}"""

    # ── LEVEL 3 ────────────────────────────────────────────────
    prompt_l3 = f"""You are an expert on building energy performance in {city}.

Predict 2 variables:
1. heated_space in m²
2. initial_heat_demand in kWh/year (TOTAL, not per m²)

REAL RATIOS heated_space / floor_area in {city.upper()}:
{hs_fa_rules}

heated_space = measured_floor_area × type_ratio
OR heated_space = floor_area × floors × factor (0.65-0.85)

REAL SPECIFIC CONSUMPTIONS (median kWh/m²/year):
{conso_table}

initial_heat_demand = heated_space × specific_consumption
Use the medians, do NOT exceed them.

RESPOND ONLY with this JSON, nothing else:
{{"heated_space": X.XX, "initial_heat_demand": X.XX}}"""

    return prompt_l1, prompt_l2, prompt_l3

# Générer les prompts dynamiques
SYSTEM_PROMPT_LEVEL1, SYSTEM_PROMPT_LEVEL2, SYSTEM_PROMPT_LEVEL3 = generate_prompts(
    CITY_CONFIG, BUILDING_STATS
)
CITY_CENTER = (CITY_CONFIG.get("center_lat", 0), CITY_CONFIG.get("center_lon", 0))

print(f"\n✓ Pipeline prête pour {CITY_CONFIG.get('city_name', 'Unknown')}")
print(f"  Prompt L1 : {(SYSTEM_PROMPT_LEVEL1)}")
#print(f"  Prompt L2 : {(SYSTEM_PROMPT_LEVEL2)}")
#print(f"  Prompt L3 : {(SYSTEM_PROMPT_LEVEL3)}")



✓ Pipeline prête pour Düsseldorf
  Prompt L1 : You are an expert on German buildings in Düsseldorf.

    Predict 3 variables:
    1. building_type from ['EFH', 'GHD', 'GMH', 'HH', 'Industrie', 'MFH', 'RH', 'Öffentlich']
    2. type_of_use from [Wohngebäude, Gewerbegebäude, Industriegebäude, Öffentliche Hand]
    3. construction_year_class from ['1860 - 1918', '1919 - 1948', '1949 - 1978', '1979 - 1986', '1987 - 1990', '1991 - 1995', '1996 - 2000', '2001 - 2004', '2005 - 2008', '2009 - 2011', '2012 - 2023']

    BUILDING TYPE DESCRIPTIONS:

    EFH (Einfamilienhaus — detached house):
    DETACHED residential house, few floors, only 1 housing unit
    In peripheral residential area, suburban neighborhood
    OSM signals: "house", "detached"

    RH (Reihenhaus — terraced house):
    SEMI-DETACHED residential house, attached to other houses in a row
    In residential housing developments, often aligned along a street
    Floor_area generally SMALLER than EFH (narrower house because it i

# Premier niveau du prompt -- 
## Prediction de | building_type | type_of_use | construction_year_class |

In [872]:
def predict_level1(profile: dict, training_examples: pd.DataFrame) -> dict:
    
    
    messages = [SystemMessage(content=SYSTEM_PROMPT_LEVEL1)]
    
    # Few-shot en CSV avec header
    header = "floor_area,apartments_min,apartments_max,construction_year,heated_space,solar_potential,building_type,type_of_use,construction_year_class"
    csv_block = header + "\n"
    
# Few-shot en messages séparés (format qui marchait le mieux avec Mistral)
    for _, row in training_examples.iterrows():
        user_msg = (
            f"floor_area: {row['floor_area']:.1f}, "
            f"apartments: {int(row['number_of_apartments_min'])}-{int(row['number_of_apartments_max'])}, "
            f"construction_year: {int(row['construction_year'])}, "
            f"heated_space: {row['heated_space']:.1f}, "
            f"solar_potential: {row.get('solar_potential', 0):.1f}"
        )
        messages.append(HumanMessage(content=user_msg))
        
        answer = json.dumps({
            "building_type": str(row['building_type']),
            "type_of_use": str(row['type_of_use']),
            "construction_year_class": str(row['construction_year_class'])
        })
        messages.append(AIMessage(content=answer))
    

    # Profil cible en CSV
    footprint = profile.get('footprint', {})
    height = profile.get('height', {})
    spatial = profile.get('spatial_context', {})
    
    target_header = "floor_area_m2,height_m,floors,compacity,tag_osm,niveaux_osm,distance_centre_km,neighboors_200m,bloc_nb_building,bloc_type_dominant,bloc_type_pct,bloc_construction_dominant,bloc_construction_pct"
    target_values = (
        f"{footprint.get('floor_area_m2', '')},"
        f"{height.get('height_p90_m', height.get('height_m', '')) if height else ''},"
        f"{height.get('estimated_floors', '') if height else ''},"
        f"{footprint.get('compactness', '')},"
        f"{footprint.get('osm_building_type', '')},"
        f"{footprint.get('osm_levels', '')},"
        f"{spatial.get('distance_center_km', '')},"
        f"{spatial.get('n_neighbors', '')},"
        f"{spatial.get('block_building_count', '')},"
        f"{spatial.get('block_dominant_building_type', '')},"
        f"{spatial.get('block_dominant_building_share', '')},"
        f"{spatial.get('block_dominant_construction_class', '')},"
        f"{spatial.get('block_dominant_construction_share', '')}"
    )
    
        # Formater les distributions pour le prompt
    bt_dist = spatial.get('block_building_type_distribution', {})
    bt_dist_str = ", ".join(f"{k}: {v}%" for k, v in sorted(bt_dist.items(), key=lambda x: -x[1]))
    
    cyc_dist = spatial.get('block_construction_year_distribution', {})
    cyc_dist_str = ", ".join(f"{k}: {v}%" for k, v in sorted(cyc_dist.items(), key=lambda x: -x[1]))
    
    target_msg = (
        f"Target building measured data: "
        f"floor_area_m2: {footprint.get('floor_area_m2', 'unknown')}, "
        f"height_m: {height.get('height_p90_m', height.get('height_m', 'unknown')) if height else 'unknown'}, "
        f"estimated_floors: {height.get('estimated_floors', 'unknown') if height else 'unknown'}, "
        f"osm_building_tag: {footprint.get('osm_building_type', 'unknown')}, "
        f"distance_center_km: {spatial.get('distance_center_km', 'unknown')}, "
        f"neighbors_200m: {spatial.get('n_neighbors', 'unknown')}, "
        f"neighbor_dominant_type: {spatial.get('neighbor_dominant_type', 'unknown')}"
        f"\n\nBlock distribution ({spatial.get('block_building_count', '?')} buildings): "
        f"\n  Building types: [{bt_dist_str}]"
        f"\n  Construction years: [{cyc_dist_str}]"
    )
    target_msg += f"\n\nPredict building_type, type_of_use, construction_year_class."
    target_msg += '\n\nRESPOND ONLY with this exact JSON, nothing else: {"building_type": "...", "type_of_use": "...", "construction_year_class": "..."}'
    
    messages.append(HumanMessage(content=target_msg))
    
    start = time.time()
    response = llm.invoke(messages)
    elapsed = time.time() - start
    
    raw = response.content.strip()
    prediction = extract_json_from_response(raw)
    
    if not prediction:
        print(f"⚠️ Aucune prédiction extraite")
    
    print(f"✓ Niveau 1 — Prédictions ({elapsed:.1f}s)")
    for k, v in prediction.items():
        print(f"  {k} : {v}")
    
    return prediction

# Deuxieme niveau du prompt -- 
## Prediction de | initial_heating_system | renovation_state | number_of_apartments_min / max |

In [873]:
def predict_level2(profile: dict, level1_predictions: dict, training_examples: pd.DataFrame) -> dict:
    messages = [SystemMessage(content=SYSTEM_PROMPT_LEVEL2)]
    
    # Few-shot en CSV
    header = "building_type,type_of_use,construction_year_class,construction_year,floor_area,heated_space,initial_heating_system,renovation_state,apartments_min,apartments_max"
    csv_block = header + "\n"
    
    for _, row in training_examples.iterrows():
        csv_block += (
            f"{row['building_type']},"
            f"{row['type_of_use']},"
            f"{row['construction_year_class']},"
            f"{int(row['construction_year'])},"
            f"{row['floor_area']:.1f},"
            f"{row['heated_space']:.1f},"
            f"{row['initial_heating_system']},"
            f"{row['renovation_state']},"
            f"{int(row['number_of_apartments_min']) if pd.notna(row['number_of_apartments_min']) else 0},"
            f"{int(row['number_of_apartments_max']) if pd.notna(row['number_of_apartments_max']) else 0}\n"
        )
    
    messages.append(HumanMessage(content=f"Dataset examples:\n{csv_block}"))
    messages.append(AIMessage(content="Understood."))
    
    # Profil cible en CSV
    footprint = profile.get('footprint', {})
    height = profile.get('height', {})
    spatial = profile.get('spatial_context', {})
    
    target_header = "building_type,type_of_use,construction_year_class,floor_area_m2,floors,distance_center_km,bloc_heating_dominant,bloc_heating_pct,bloc_renovation_dominant,bloc_renovation_pct"
    target_values = (
        f"{level1_predictions.get('building_type', '')},"
        f"{level1_predictions.get('type_of_use', '')},"
        f"{level1_predictions.get('construction_year_class', '')},"
        f"{footprint.get('floor_area_m2', '')},"
        f"{height.get('estimated_floors', '') if height else ''},"
        f"{spatial.get('distance_center_km', '')},"
        f"{spatial.get('block_dominant_heating', '')},"
        f"{spatial.get('block_dominant_heating_share', '')},"
        f"{spatial.get('block_dominant_renovation', '')},"
        f"{spatial.get('block_dominant_renovation_share', '')}"
    )
    
    target_msg = f"Target building:\n{target_header}\n{target_values}"
    target_msg += f"\n\nPredict initial_heating_system, renovation_state, number_of_apartments_min, number_of_apartments_max."
    target_msg += '\n\nRESPOND ONLY with this exact JSON, nothing else: {"initial_heating_system": "...", "renovation_state": "...", "number_of_apartments_min": X, "number_of_apartments_max": Y}'
    
    messages.append(HumanMessage(content=target_msg))
    
    start = time.time()
    response = llm.invoke(messages)
    elapsed = time.time() - start
    
    raw = response.content.strip()
    prediction = extract_json_from_response(raw)
    
    if not prediction:
        print(f"⚠️ Aucune prédiction extraite")
    
    print(f"✓ Niveau 2 — Prédictions ({elapsed:.1f}s)")
    for k, v in prediction.items():
        print(f"  {k} : {v}")
    
    return prediction

# Troisieme niveau du prompt -- 
## Prediction de | heated_space | initial_heat_demand | 

In [874]:
def predict_level3(profile: dict, level1_preds: dict, level2_preds: dict, 
                   training_examples: pd.DataFrame) -> dict:
    messages = [SystemMessage(content=SYSTEM_PROMPT_LEVEL3)]
    
    # Few-shot en CSV
    header = "building_type,construction_year_class,renovation_state,initial_heating_system,floor_area,construction_year,apartments_min,apartments_max,heated_space,initial_heat_demand"
    csv_block = header + "\n"
    
    for _, row in training_examples.iterrows():
        csv_block += (
            f"{row['building_type']},"
            f"{row['construction_year_class']},"
            f"{row['renovation_state']},"
            f"{row['initial_heating_system']},"
            f"{row['floor_area']:.1f},"
            f"{int(row['construction_year'])},"
            f"{int(row['number_of_apartments_min']) if pd.notna(row['number_of_apartments_min']) else 0},"
            f"{int(row['number_of_apartments_max']) if pd.notna(row['number_of_apartments_max']) else 0},"
            f"{row['heated_space']:.2f},"
            f"{row['initial_heat_demand']:.2f}\n"
        )
    
    messages.append(HumanMessage(content=f"Dataset examples:\n{csv_block}"))
    messages.append(AIMessage(content="Understood."))
    
    # Profil cible en CSV
    footprint = profile.get('footprint', {})
    height = profile.get('height', {})
    
    target_header = "building_type,construction_year_class,renovation_state,initial_heating_system,floor_area_m2,floors,apartments_min,apartments_max"
    target_values = (
        f"{level1_preds.get('building_type', '')},"
        f"{level1_preds.get('construction_year_class', '')},"
        f"{level2_preds.get('renovation_state', '')},"
        f"{level2_preds.get('initial_heating_system', '')},"
        f"{footprint.get('floor_area_m2', '')},"
        f"{height.get('estimated_floors', '') if height else ''},"
        f"{level2_preds.get('number_of_apartments_min', '')},"
        f"{level2_preds.get('number_of_apartments_max', '')}"
    )
# Pré-calculer la consommation spécifique pour aider le modèle
    cyc = level1_preds.get('construction_year_class', '')
    rs = level2_preds.get('renovation_state', 'not_renovated')
    conso = CITY_CONFIG.get('conso_specifique', {}).get(cyc, {}).get(rs, 100)
    
    target_header = "building_type,construction_year_class,renovation_state,initial_heating_system,floor_area_m2,floors,apartments_min,apartments_max,conso_specifique_kwh_m2"
    target_values = (
        f"{level1_preds.get('building_type', '')},"
        f"{level1_preds.get('construction_year_class', '')},"
        f"{level2_preds.get('renovation_state', '')},"
        f"{level2_preds.get('initial_heating_system', '')},"
        f"{footprint.get('floor_area_m2', '')},"
        f"{height.get('estimated_floors', '') if height else ''},"
        f"{level2_preds.get('number_of_apartments_min', '')},"
        f"{level2_preds.get('number_of_apartments_max', '')},"
        f"{conso}"
    )
    
    target_msg = f"Target building (predict heated_space and initial_heat_demand):\n{target_header}\n{target_values}"
    target_msg += f"\n\nReminder: initial_heat_demand = heated_space × specific_conso_kwh_m2 ({conso} kWh/m²/year for this class/renovation combination)"
    target_msg += '\n\nRESPOND ONLY with this exact JSON, nothing else: {"heated_space": X.XX, "initial_heat_demand": X.XX}'
    
    messages.append(HumanMessage(content=target_msg))
    
    start = time.time()
    response = llm.invoke(messages)
    elapsed = time.time() - start
    
    raw = response.content.strip()
    prediction = extract_json_from_response(raw)
    
    if not prediction:
        print(f"⚠️ Aucune prédiction extraite")
    
    print(f"✓ Niveau 3 — Prédictions ({elapsed:.1f}s)")
    for k, v in prediction.items():
        print(f"  {k} : {v}")
    
    return prediction

## PIPELINE COMPLÈTE — Les 3 niveaux en cascade

In [ ]:
# =============================================================================
# PIPELINE COMPLÈTE — Les 3 niveaux en cascade
# =============================================================================

def run_full_prediction(profile: dict, training_examples: pd.DataFrame) -> dict:
    """
    Exécute la prédiction en cascade sur les 3 niveaux.
    Retourne toutes les prédictions consolidées.
    """
    print("=" * 60)
    print("PRÉDICTION LLM EN CASCADE")
    print("=" * 60)
    
    # Niveau 1
    print("\n── Niveau 1 : Type, Usage, Construction ──")
    level1 = predict_level1(profile, training_examples)
    2  # Rate limiting
    
    # Niveau 2
    print("\n── Niveau 2 : Chauffage, Rénovation, Appartements ──")
    level2 = predict_level2(profile, level1, training_examples)
    time.sleep(2)
    
    # Niveau 3
    print("\n── Niveau 3 : Surface chauffée, Demande thermique ──")
    level3 = predict_level3(profile, level1, level2, training_examples)
    
    # Consolider
    all_predictions = {**level1, **level2, **level3}
    
    print("\n" + "=" * 60)
    print("PRÉDICTIONS CONSOLIDÉES")
    print("=" * 60)
    for k, v in all_predictions.items():
        print(f"  {k} : {v}")
    
    return all_predictions

In [876]:
# ── Lancer la prédiction sur le bâtiment de test ──────────────
if profile:
    predictions = run_full_prediction(profile, training_examples)

PRÉDICTION LLM EN CASCADE

── Niveau 1 : Type, Usage, Construction ──
✓ Niveau 1 — Prédictions (0.6s)
  building_type : MFH
  type_of_use : Wohngeb e4ude
  construction_year_class : 1919 - 1948

── Niveau 2 : Chauffage, Rénovation, Appartements ──
✓ Niveau 2 — Prédictions (0.5s)
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : not_renovated
  number_of_apartments_min : 7
  number_of_apartments_max : 12

── Niveau 3 : Surface chauffée, Demande thermique ──
✓ Niveau 3 — Prédictions (0.5s)
  heated_space : 346.3
  initial_heat_demand : 36361.5

PRÉDICTIONS CONSOLIDÉES
  building_type : MFH
  type_of_use : Wohngeb e4ude
  construction_year_class : 1919 - 1948
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : not_renovated
  number_of_apartments_min : 7
  number_of_apartments_max : 12
  heated_space : 346.3
  initial_heat_demand : 36361.5


## Comparaison Prédiction VS Digital Twin Values

In [877]:
def compare_predictions(predictions: dict, ground_truth: pd.Series) -> pd.DataFrame:
    if ground_truth is None:
        print("⚠️ Pas de ground truth disponible")
        return None
    
    rows = []
    
    categorical_vars = ['building_type', 'type_of_use', 'construction_year_class',
                        'initial_heating_system', 'renovation_state']
    integer_vars = ['number_of_apartments_min', 'number_of_apartments_max']
    numeric_vars = ['heated_space', 'initial_heat_demand']
    
    all_vars = categorical_vars + integer_vars + numeric_vars
    
    for var in all_vars:
        pred = predictions.get(var, None)
        actual = ground_truth.get(var, None) if var in ground_truth.index else None
        
        # Vérifier que les deux valeurs existent
        if pred is None or actual is None or (isinstance(actual, float) and pd.isna(actual)):
            rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                        'match': '?', 'erreur_%': ''})
            continue
        
        if var in categorical_vars:
            pred_str = str(pred).strip()
            actual_str = str(actual).strip()
            match = '✓' if pred_str == actual_str else '✗'
            rows.append({'variable': var, 'prédit': pred_str, 'réel': actual_str, 
                        'match': match, 'erreur_%': ''})
        
        elif var in integer_vars:
            try:
                match = '✓' if int(float(pred)) == int(float(actual)) else '✗'
                rows.append({'variable': var, 'prédit': str(int(float(pred))), 
                            'réel': str(int(float(actual))), 'match': match, 'erreur_%': ''})
            except (ValueError, TypeError):
                rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                            'match': '?', 'erreur_%': ''})
        
        elif var in numeric_vars:
            try:
                pred_f = float(pred)
                actual_f = float(actual)
                if actual_f != 0:
                    error_pct = abs(pred_f - actual_f) / abs(actual_f) * 100
                else:
                    error_pct = 0
                match = '✓' if error_pct < 20 else '~' if error_pct < 50 else '✗'
                rows.append({'variable': var, 'prédit': f"{pred_f:.1f}", 
                            'réel': f"{actual_f:.1f}", 'match': match, 
                            'erreur_%': f"{error_pct:.1f}"})
            except (ValueError, TypeError):
                rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                            'match': '?', 'erreur_%': ''})
    
    df_comparison = pd.DataFrame(rows)
    
    # Résumé
    cat_results = df_comparison[df_comparison['variable'].isin(categorical_vars)]
    cat_correct = (cat_results['match'] == '✓').sum()
    cat_total = (cat_results['match'].isin(['✓', '✗'])).sum()
    
    int_results = df_comparison[df_comparison['variable'].isin(integer_vars)]
    int_correct = (int_results['match'] == '✓').sum()
    int_total = (int_results['match'].isin(['✓', '✗'])).sum()
    
    num_results = df_comparison[df_comparison['variable'].isin(numeric_vars)]
    num_errors = num_results['erreur_%'].replace('', np.nan).dropna().astype(float)
    
    print("=" * 70)
    print("COMPARAISON PRÉDICTIONS vs GROUND TRUTH")
    print("=" * 70)
    print(df_comparison.to_string(index=False))
    print(f"\nVariables catégorielles : {cat_correct}/{cat_total} correctes")
    print(f"Variables entières : {int_correct}/{int_total} correctes")
    if len(num_errors) > 0:
        print(f"Variables numériques : erreur moyenne = {num_errors.mean():.1f}%")
    
    return df_comparison

# ── Comparaison ───────────────────────────────────────────────
if profile:
    ground_truth = find_ground_truth(profile, df_individual)
    comparison = compare_predictions(predictions, ground_truth)

  Adresse recherchée : 'Fleher Straße 177'
  Normalisée : 'fleher strasse 177'

✓ Match exact trouvé !
  Adresse dataset : Fleher Straße 177
  floor_area → OSM: 159.6 m² | Dataset: 159.6 m² | Écart: 0.0 m²
  building_type : MFH
  heated_space : 201.1243871744589
  initial_heat_demand : 38066.4600296
  construction_year : 1978.0
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : partially_renovated
COMPARAISON PRÉDICTIONS vs GROUND TRUTH
                variable              prédit                réel match erreur_%
           building_type                 MFH                 MFH     ✓         
             type_of_use       Wohngeb e4ude         Wohngebäude     ✗         
 construction_year_class         1919 - 1948         1949 - 1978     ✗         
  initial_heating_system Gaskessel (Bestand) Gaskessel (Bestand)     ✓         
        renovation_state       not_renovated partially_renovated     ✗         
number_of_apartments_min                   7                  

## Mémoire sur les infos profile des batiments étudiés (OSM requetes)

In [878]:
import hashlib

PROFILE_CACHE = {}

def collect_building_profile_cached(address: str, df_blocks: pd.DataFrame, tif_path: str = None) -> dict:
    """
    Wrapper avec cache — évite de refaire les requêtes OSM si le profil existe déjà.
    """
    cache_key = hashlib.md5(address.encode()).hexdigest()
    
    if cache_key in PROFILE_CACHE:
        print(f"✓ Profil chargé depuis le cache : {address}")
        return PROFILE_CACHE[cache_key]
    
    profile = collect_building_profile(address, df_blocks, tif_path)
    
    if profile is not None:
        PROFILE_CACHE[cache_key] = profile
        print(f"✓ Profil sauvegardé dans le cache")
    
    return profile


def save_cache_to_disk(filepath: str = "profile_cache.json"):
    """Sauvegarde le cache sur disque pour persister entre les sessions."""
    cache_serializable = {}
    for key, profile in PROFILE_CACHE.items():
        cache_serializable[key] = json.loads(json.dumps(profile, default=str))
    
    with open(filepath, 'w') as f:
        json.dump(cache_serializable, f, indent=2, ensure_ascii=False)
    print(f"✓ Cache sauvegardé : {len(cache_serializable)} profils → {filepath}")


def load_cache_from_disk(filepath: str = "profile_cache.json"):
    """Charge le cache depuis le disque."""
    global PROFILE_CACHE
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            PROFILE_CACHE = json.load(f)
        print(f"✓ Cache chargé : {len(PROFILE_CACHE)} profils depuis {filepath}")
    else:
        print("ℹ️ Pas de cache existant")

## Trouver 10 adresses utilisables pour les testes dans Digital Twin Measured

In [747]:
# Sélectionner 5 adresses de test variées dans Gelsenkirchen
test_candidates = df_individual.dropna(subset=['street', 'heated_space', 'initial_heat_demand'])

# Stratifier par building_type
test_addresses = []
for bt, group in test_candidates.groupby('building_type'):
    if len(group) > 0:
        sample = group.sample(n=min(2, len(group)), random_state=200)
        for _, row in sample.iterrows():
            street = row['street']
            # Reconstruire l'adresse complète
            address = f"{street}, Düsseldorf, Germany"
            test_addresses.append(address)
            if len(test_addresses) >= 5:
                break
    if len(test_addresses) >= 5:
        break

print(f"Adresses de test ({len(test_addresses)}) :")
for i, addr in enumerate(test_addresses):
    print(f"  {i+1}. {addr}")

TEST_ADDRESSES = test_addresses

Adresses de test (5) :
  1. Niederrheinstraße 245, Düsseldorf, Germany
  2. Ilvericher Straße 11, Düsseldorf, Germany
  3. Harkortstraße 7, Düsseldorf, Germany
  4. Ratinger Straße, Düsseldorf, Germany
  5. Cranachstraße 21, Düsseldorf, Germany


## Test sur 10 adresses du dataset + Comparaison

In [879]:
# =============================================================================
# TEST BATCH — 10 adresses pour calibration
# =============================================================================

# ── LISTE D'ADRESSES DE TEST ──────────────────────────────────
# Remplace par des adresses de ton dataset pour lesquelles tu as la ground truth
TEST_ADDRESSES = [
    "Gräulinger Straße 117, Düsseldorf, Germany", 
    "Franz-Vaahsen-Weg 14, Düsseldorf, Germany", 
    "Malkastenstraße 7, Düsseldorf, Germany",
    "In der Gartenstadt 77, Düsseldorf, Germany", 
    "Willi-Terbuyken-Straße 18, Düsseldorf, Germany",
] 

load_cache_from_disk()

def run_batch_test(addresses: list, df_blocks: pd.DataFrame, df_individual: pd.DataFrame,
                   training_examples: pd.DataFrame, tif_folder: str = None) -> pd.DataFrame:
    """
    Lance le pipeline complet sur plusieurs adresses et compile les résultats.
    """
    all_results = []
    
    for i, address in enumerate(addresses):
        print(f"\n{'#' * 70}")
        print(f"# BÂTIMENT {i+1}/{len(addresses)} : {address}")
        print(f"{'#' * 70}")
        
        try:
            # Phase 1 — Collecte
            prof = collect_building_profile_cached(address, df_blocks, tif_folder)
            
            if prof is None:
                print(f"⚠️ Profil non disponible pour {address}")
                continue
            
            # Phase 2 — Prédiction
            preds = run_full_prediction(prof, training_examples)
            
            # Phase 3 — Ground truth
            gt = find_ground_truth(prof, df_individual)
            
            if gt is not None:
                result = {'address': address}
                for var in ['building_type', 'type_of_use', 'construction_year_class',
                           'initial_heating_system', 'renovation_state',
                           'heated_space', 'initial_heat_demand']:
                    result[f'{var}_pred'] = preds.get(var)
                    result[f'{var}_real'] = gt.get(var)
                    
                    # Calculer l'erreur pour les numériques
                    if var in ['heated_space', 'initial_heat_demand']:
                        try:
                            p, r = float(preds.get(var, 0)), float(gt.get(var, 0))
                            result[f'{var}_error_%'] = round(abs(p-r)/r*100, 1) if r != 0 else None
                        except (ValueError, TypeError):
                            result[f'{var}_error_%'] = None
                    else:
                        result[f'{var}_match'] = str(preds.get(var, '')).strip() == str(gt.get(var, '')).strip()
                
                all_results.append(result)
            
            # Pause entre les adresses (rate limiting OSM + Mistral)
            time.sleep(5)
            
        except Exception as e:
            print(f"❌ Erreur pour {address}: {e}")
            continue
    
    if not all_results:
        print("❌ Aucun résultat")
        return None
    
    df_results = pd.DataFrame(all_results)
    
    
    # ── Résumé global ──────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RÉSUMÉ BATCH")
    print("=" * 70)
    
    save_cache_to_disk()

    for var in ['building_type', 'type_of_use', 'construction_year_class',
               'initial_heating_system', 'renovation_state']:
        col = f'{var}_match'
        if col in df_results.columns:
            accuracy = df_results[col].mean() * 100
            print(f"  {var} : {accuracy:.0f}% correct")
    
    for var in ['heated_space', 'initial_heat_demand']:
        col = f'{var}_error_%'
        if col in df_results.columns:
            mean_err = df_results[col].dropna().mean()
            print(f"  {var} : erreur moyenne {mean_err:.1f}%")
    
    return df_results

#── Lancer le test batch (décommenter quand les adresses sont remplies) ──
batch_results = run_batch_test(
    TEST_ADDRESSES, df_blocks, df_individual, training_examples, TIF_FOLDER
)

✓ Cache chargé : 61 profils depuis profile_cache.json

######################################################################
# BÂTIMENT 1/5 : Gräulinger Straße 117, Düsseldorf, Germany
######################################################################
✓ Profil chargé depuis le cache : Gräulinger Straße 117, Düsseldorf, Germany
PRÉDICTION LLM EN CASCADE

── Niveau 1 : Type, Usage, Construction ──
✓ Niveau 1 — Prédictions (0.6s)
  building_type : MFH
  type_of_use : Wohngebäude
  construction_year_class : 1979 - 1986

── Niveau 2 : Chauffage, Rénovation, Appartements ──
✓ Niveau 2 — Prédictions (0.6s)
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : partially_renovated
  number_of_apartments_min : 7
  number_of_apartments_max : 12

── Niveau 3 : Surface chauffée, Demande thermique ──
✓ Niveau 3 — Prédictions (0.6s)
  heated_space : 193.79
  initial_heat_demand : 22478.64

PRÉDICTIONS CONSOLIDÉES
  building_type : MFH
  type_of_use : Wohngebäude
  construction_year

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# SÉLECTION D'ADRESSES DE TEST PAR BUILDING_TYPE
# =============================================================================

def select_test_addresses_by_type(df: pd.DataFrame, n_per_type: int = 2, seed: int = 42) -> pd.DataFrame:
    """
    Sélectionne des adresses de test variées pour chaque building_type.
    Pour chaque type, prend un petit et un grand bâtiment (par floor_area).
    """
    np.random.seed(seed)
    
    test_buildings = []
    
    for bt in sorted(df['building_type'].dropna().unique()):
        subset = df[df['building_type'] == bt].dropna(subset=['street', 'floor_area'])
        # Filtrer les adresses sans numéro (doit contenir au moins un chiffre)
        subset = subset[subset['street'].str.contains(r'\d', na=False)]
        
        if len(subset) < 2:
            test_buildings.append(subset)
            continue
        
        # Prendre un petit (p25) et un grand (p75) par floor_area
        small = subset[subset['floor_area'] <= subset['floor_area'].quantile(0.25)].sample(n=1, random_state=seed)
        large = subset[subset['floor_area'] >= subset['floor_area'].quantile(0.75)].sample(n=1, random_state=seed)
        
        test_buildings.append(pd.concat([small, large]))
    
    result = pd.concat(test_buildings)
    
    print(f"✓ {len(result)} bâtiments sélectionnés pour le test")
    print(f"{'='*80}")
    for bt in sorted(result['building_type'].unique()):
        subset = result[result['building_type'] == bt]
        for _, row in subset.iterrows():
            print(f"  {bt:12s} | {row['street']:40s} | floor_area: {row['floor_area']:.1f}m² | year: {int(row['construction_year'])}")
    
    return result


# Lancer la sélection
test_df = select_test_addresses_by_type(df_individual, n_per_type=2)

✓ 16 bâtiments sélectionnés pour le test
  EFH          | Ohmweg 1a                                | floor_area: 54.2m² | year: 1984
  EFH          | Weilburger Weg 35                        | floor_area: 89.5m² | year: 1956
  GHD          | Immermannstraße 15                       | floor_area: 372.2m² | year: 1977
  GHD          | Heinrich-Heine-Allee 12                  | floor_area: 992.5m² | year: 1971
  GMH          | Bahnstraße 76                            | floor_area: 279.1m² | year: 1949
  GMH          | Velberter Straße 38                      | floor_area: 642.4m² | year: 1962
  HH           | Tersteegenstraße 17                      | floor_area: 264.3m² | year: 1960
  HH           | Tersteegenstraße 63                      | floor_area: 332.2m² | year: 2009
  Industrie    | Gaußstraße 23                            | floor_area: 158.1m² | year: 1908
  Industrie    | Pinienstraße 19                          | floor_area: 641.7m² | year: 1961
  MFH          | Seydlitzstraße

In [889]:
# =============================================================================
# TEST NIVEAU 1 UNIQUEMENT — BUILDING TYPE
# =============================================================================

def test_level1_only(test_df: pd.DataFrame, df_blocks: pd.DataFrame, 
                     training_examples: pd.DataFrame, tif_folder: str = None):
    """
    Lance le profiling + prédiction Niveau 1 uniquement sur les bâtiments de test.
    Compare et analyse les features discriminantes.
    """
    results = []
    
    for i, (idx, row) in enumerate(test_df.iterrows()):
        street = row['street']
        city = CITY_CONFIG.get('city_name', 'Germany')
        address = f"{street}, {city}, Germany"
        
        print(f"\n{'#'*70}")
        print(f"# [{i+1}/{len(test_df)}] {row['building_type']} — {address}")
        print(f"{'#'*70}")
        
        try:
            # Profiling (avec cache)
            prof = collect_building_profile_cached(address, df_blocks, tif_folder)
            
            if prof is None:
                print("⚠️ Profil non disponible")
                continue
            
            # Prédiction Niveau 1 uniquement
            print("\n── Prédiction Niveau 1 ──")
            level1 = predict_level1(prof, training_examples)
            
            # Collecter les résultats
            footprint = prof.get('footprint', {})
            height = prof.get('height', {})
            spatial = prof.get('spatial_context', {})
            
            result = {
                'address': address,
                'real_building_type': row['building_type'],
                'pred_building_type': level1.get('building_type', '?'),
                'match': '✓' if level1.get('building_type') == row['building_type'] else '✗',
                'floor_area': footprint.get('floor_area_m2'),
                'height_m': height.get('height_p90_m', height.get('height_m')) if height else None,
                'estimated_floors': height.get('estimated_floors') if height else None,
                'osm_tag': footprint.get('osm_building_type'),
                'distance_center_km': spatial.get('distance_center_km'),
                'n_neighbors': spatial.get('n_neighbors'),
                'neighbor_dominant': spatial.get('neighbor_dominant_type'),
                'block_count': spatial.get('block_building_count'),
                'block_dominant_type': spatial.get('block_dominant_building_type'),
                'block_dominant_share': spatial.get('block_dominant_building_share'),
                'block_bt_distribution': spatial.get('block_building_type_distribution', {}),
            }
            
            results.append(result)
            
            # Affichage compact
            match_icon = result['match']
            print(f"\n  {match_icon} Réel: {result['real_building_type']} | Prédit: {result['pred_building_type']}")
            print(f"    floor_area={result['floor_area']}m² | height={result['height_m']}m | floors={result['estimated_floors']}")
            print(f"    osm_tag={result['osm_tag']} | neighbors={result['n_neighbors']} | block_dominant={result['block_dominant_type']}({result['block_dominant_share']})")
            
            time.sleep(2)
            
        except Exception as e:
            print(f"❌ Erreur: {e}")
            continue
    
    # Sauvegarder le cache
    save_cache_to_disk()
    
    return results


# Lancer le test
results_l1 = test_level1_only(test_df, df_blocks, training_examples, TIF_FOLDER)


######################################################################
# [1/16] EFH — Ohmweg 1a, Düsseldorf, Germany
######################################################################
PROFIL BÂTIMENT : Ohmweg 1a, Düsseldorf, Germany

── Étape 1 : Géocodage ──
✓ Géocodage réussi
  Adresse : 1a, Ohmweg, Wersten-West, Wersten, Stadtbezirk 9, Düsseldorf, Nordrhein-Westfalen, 40591, Deutschland
  Coordonnées : (51.186911, 6.809834)

── Étape 2 : Empreinte bâtiment (dataset individuel) ──
  Surface au sol : 54.2 m²

── Étape 3 : Hauteur (TIF) ──
  240 tuiles TIF disponibles
  ✓ Tuile trouvée : ndom50_32346_5672_1_nw_2023.tif
  Hauteur max : 3.3 m
  Hauteur p90 : 3.2 m
  Étages estimés : 1
  Résolution : 0.5m/pixel

── Étape 4 : Identification du bloc ──
✓ Bloc identifié
  Floor ID : 31be6c17-e597-46ab-a1df-c25b16f54e26
  Nom : 2
  Nombre de bâtiments : 354
  Répartition building_type :
    EFH: 31.1%
    Industrie: 2.5%
    MFH: 26.3%
    RH: 39.3%

── Étape 5 : Contexte spatial ──
  Ré

In [890]:
# =============================================================================
# ANALYSE DES FEATURES DISCRIMINANTES
# =============================================================================

def analyze_level1_results(results: list):
    """
    Analyse les résultats du test Niveau 1 pour identifier
    quelles features aident ou nuisent à la prédiction.
    """
    df_r = pd.DataFrame(results)
    
    print("=" * 80)
    print("ANALYSE DES RÉSULTATS NIVEAU 1")
    print("=" * 80)
    
    # Taux de succès global
    correct = (df_r['match'] == '✓').sum()
    total = len(df_r)
    print(f"\nTaux de succès : {correct}/{total} ({correct/total*100:.0f}%)")
    
    # Succès par type réel
    print(f"\nPar building_type réel :")
    for bt in sorted(df_r['real_building_type'].unique()):
        subset = df_r[df_r['real_building_type'] == bt]
        ok = (subset['match'] == '✓').sum()
        print(f"  {bt:12s} : {ok}/{len(subset)} correct")
    
    # Analyse des erreurs
    errors = df_r[df_r['match'] == '✗']
    if len(errors) > 0:
        print(f"\n{'='*80}")
        print(f"ANALYSE DES ERREURS ({len(errors)} cas)")
        print(f"{'='*80}")
        
        for _, row in errors.iterrows():
            print(f"\n  {row['real_building_type']} prédit comme {row['pred_building_type']}")
            print(f"    floor_area: {row['floor_area']}m²")
            print(f"    height: {row['height_m']}m, floors: {row['estimated_floors']}")
            print(f"    osm_tag: {row['osm_tag']}")
            print(f"    distance_center: {row['distance_center_km']}km")
            print(f"    neighbors: {row['n_neighbors']}, neighbor_dominant: {row['neighbor_dominant']}")
            print(f"    block: {row['block_count']} buildings, dominant: {row['block_dominant_type']} ({row['block_dominant_share']})")
            print(f"    block distribution: {row['block_bt_distribution']}")
    
    # Comparaison features correctes vs erreurs
    print(f"\n{'='*80}")
    print(f"FEATURES MOYENNES : CORRECT vs ERREUR")
    print(f"{'='*80}")
    
    for col in ['floor_area', 'height_m', 'estimated_floors', 'distance_center_km', 'n_neighbors', 'block_count']:
        correct_vals = df_r[df_r['match'] == '✓'][col].dropna()
        error_vals = df_r[df_r['match'] == '✗'][col].dropna()
        if len(correct_vals) > 0 and len(error_vals) > 0:
            print(f"  {col:25s} : correct={correct_vals.median():.1f} | erreur={error_vals.median():.1f}")
    
    # OSM tags pour correct vs erreur
    print(f"\n  OSM tags (correct) : {df_r[df_r['match']=='✓']['osm_tag'].value_counts().to_dict()}")
    print(f"  OSM tags (erreur)  : {df_r[df_r['match']=='✗']['osm_tag'].value_counts().to_dict()}")
    
    return df_r


# Lancer l'analyse
df_analysis = analyze_level1_results(results_l1)

ANALYSE DES RÉSULTATS NIVEAU 1

Taux de succès : 8/14 (57%)

Par building_type réel :
  EFH          : 1/2 correct
  GHD          : 1/2 correct
  GMH          : 1/2 correct
  HH           : 2/2 correct
  Industrie    : 1/2 correct
  MFH          : 2/2 correct
  Öffentlich   : 0/2 correct

ANALYSE DES ERREURS (6 cas)

  EFH prédit comme RH
    floor_area: 54.20065400071324m²
    height: 3.25m, floors: 1
    osm_tag: None
    distance_center: 5.19km
    neighbors: 199, neighbor_dominant: house
    block: 354 buildings, dominant: None (None)
    block distribution: {'EFH': 31.1, 'Industrie': 2.5, 'MFH': 26.3, 'RH': 39.3}

  GHD prédit comme HH
    floor_area: 992.5045709367458m²
    height: 26.76m, floors: 8
    osm_tag: None
    distance_center: 0.2km
    neighbors: 269, neighbor_dominant: yes
    block: 405 buildings, dominant: None (None)
    block distribution: {'EFH': 26.4, 'GHD': 18.0, 'GMH': 2.7, 'MFH': 34.8, 'RH': 10.9, 'Öffentlich': 6.9}

  GMH prédit comme MFH
    floor_area: 64